# IntegrationPlotModify

Set the paths marked in the code before running this notebook.

Reproduce the seven final Fig1 PDFs from the saved integration intermediates. No integration, clustering or UMAP is recomputed. All plotting and editable-PDF export functions are included in this notebook; the old Python scripts and existing PDFs are not runtime inputs.

Use the **scvi_env** Python kernel and Windows with Adobe Illustrator installed. Run all cells in order. The default output folder is the configured directory and the seven same-name PDFs are replaced when the plotting cells run. Source H5AD, model and NPZ files are read-only.

The saved `Fig1-plot.npz` is required: the postnatal H5AD contains `X_scvi` but no `X_umap`. Its frozen postnatal coordinates preserve the original 985-cell figure. Embryonic and whole-period coordinates are checked against their H5AD files. Label positions and real-cell leader anchors are frozen to the final layout.

Parameters are beside the corresponding plotting steps. `UMAP_` and `DOT_` prefixes separate the two original scripts' global names without changing their values or methods. To deliberately change label layout, set `UMAP_USE_SAVED_LAYOUT=False`; otherwise the saved positions remain fixed. Validation uses a temporary output override, so the current PDFs remain untouched during this notebook's verification.

In [ ]:
#1. =========== Paths and imports
# Replace each ROOT placeholder with the required file or directory path before running.
from pathlib import Path
import base64
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
import textwrap

ROOT = Path(".")  # Set the project root before running this template.
# Set the output directory path.
OUTPUT_DIR = ROOT
# Set the Arial font file path.
FONT_PATH = ROOT
ORIGINAL_TEMP_ENV = {key:os.environ.get(key) for key in ("TMP","TEMP","MPLCONFIGDIR")}
# Set the temporary working directory or file path.
WORK_ROOT = ROOT
WORK_ROOT.mkdir(parents=True, exist_ok=True)
TASK_TMP = Path(tempfile.mkdtemp(prefix="run-", dir=WORK_ROOT))
# Set the temporary working directory or file path.
os.environ["MPLCONFIGDIR"] = str(ROOT)
os.environ["TMP"] = os.environ["TEMP"] = str(TASK_TMP)
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True

import h5py
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import colors, font_manager
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator, FormatStrFormatter
from matplotlib.transforms import Bbox
from anndata._io.specs import read_elem
from scipy.special import ndtr
from scipy.stats import rankdata
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
try:
    from pypdf import PdfReader, PdfWriter
    from pypdf.generic import ContentStream, NameObject
except ImportError:
    
    from pypdf import PdfReader, PdfWriter
    from pypdf.generic import ContentStream, NameObject

assert ROOT.is_dir() and FONT_PATH.is_file()
font_manager.fontManager.addfont(str(FONT_PATH))
pdfmetrics.registerFont(TTFont("Arial", str(FONT_PATH)))
plt.rcParams.update({"font.family":"Arial", "axes.unicode_minus":False,
                     "axes.grid":False, "text.usetex":False})

print("Output:", OUTPUT_DIR)

## 2. Read the integration intermediates
Read only the required H5AD observations and coordinates. Retain original cell order, labels, stages and the approved shared palette. The coordinate archive also carries the final label layout.

In [ ]:
#2. =========== Load and check integration inputs
# Set the data file path; uses Fig1-plot.npz from final Fig1 plot preparation.
UMAP_STATE_PATH = ROOT
# Set each stage input path; uses {stage}_Embryonic_Inte_adata.h5ad generated by the corresponding fig1_E*_clustering.ipynb notebook.
# Set the result file path; uses SCVI_10000_Postnatal_Inte_adata.h5ad generated by postnatal scVI integration.
# Set the result file path; uses Whole_CSS.h5ad generated by whole-period CSS integration.
UMAP_INPUTS = {
    "Embryonic": (ROOT, "stage_celltype", "stage_standardized", 7355),
    "Postnatal": (ROOT, None, "stage_analysis", 985),
    "Whole_Period": (ROOT, "source_stage_celltype", "input_stage", 8340),
}
with np.load(UMAP_STATE_PATH, allow_pickle=False) as saved:
    umap_arrays = {key:saved[key].copy() for key in saved.files if key != "metadata_json"}
    umap_metadata = json.loads(str(saved["metadata_json"]))
for dataset,(relative,label_key,stage_key,expected_n) in UMAP_INPUTS.items():
    # Set the data file or directory path.
    path = ROOT
    with h5py.File(path,"r") as handle:
        obs = read_elem(handle["obs"])
        labels = (obs[stage_key].astype(str)+" | "+obs["cell_type"].astype(str)
                  if label_key is None else obs[label_key].astype(str))
        assert len(obs) == expected_n and obs.index.is_unique
        assert np.array_equal(umap_arrays[dataset+"_cell_ids"],np.asarray(obs.index.astype(str),dtype=str))
        assert np.array_equal(umap_arrays[dataset+"_cell"],np.asarray(labels,dtype=str))
        assert np.array_equal(umap_arrays[dataset+"_stage"],np.asarray(obs[stage_key].astype(str),dtype=str))
        xy = umap_arrays[dataset+"_xy"]
        assert xy.shape == (expected_n,2) and np.isfinite(xy).all()
        if "X_umap" in handle["obsm"]:
            assert np.array_equal(xy,handle["obsm/X_umap"][:])
    print(f"{dataset}: {expected_n:,} cells; original IDs, labels and coordinates verified.")
del obs, labels

## 3. UMAP appearance
These are the approved values. Embryonic and postnatal axes have the same source size. Whole-period typography and line widths account for the screenshot placement ratio, 725/475. One complete legend entry is one text frame; line breaks inside a label remain inside that label’s text frame.

In [ ]:
#3. =========== UMAP plotting parameters
UMAP_USE_SAVED_LAYOUT = True
UMAP_RANDOM_SEED = 0

# Selected 36-label palette, adapted from Yao et al., Nature 2023
# (10.1038/s41586-023-06812-z), with adjusted lightness and chroma.
# Use this palette for all cell plots; retain the original palette for PDF recovery.
UMAP_CELL_ANNOTATION_COLORS = {
    "E9.5 | CVG neuroblast": "#347659",
    "E10.5 | O-2: delaminating otic/CVG neuroblast": "#B24F18",
    "E10.5 | O-3: developing CVG neuron (not SGN-specific)": "#2998AD",
    "E11.5 | Undifferentiated / proliferative CVG": "#8888E8",
    "E11.5 | Differentiating SGN": "#9F4EB2",
    "E13.5 | Late CVG": "#95AA01",
    "E13.5 | Differentiating SGN": "#74AB63",
    "E14.5 | Type Ia + Type II precursor candidate": "#B12071",
    "E14.5 | Type Ib + Type Ic precursor candidate": "#E86765",
    "E14.5 | Undifferentiated SGN": "#E49562",
    "E15.5 | Type Ia + Type II precursor candidate": "#AC8F65",
    "E15.5 | Type Ib + Type Ic precursor candidate": "#246279",
    "E15.5 | Undifferentiated SGN": "#AD870E",
    "E16.5 | Emerging Type Ic": "#E1B338",
    "E16.5 | Type Ia + Type II precursor candidate": "#4C7BB0",
    "E16.5 | Type Ib + Type Ic precursor candidate": "#6671EC",
    "E16.5 | Undifferentiated SGN": "#09770F",
    "E17.5 | Type Ib + Type Ic precursor candidate": "#84AAE5",
    "E17.5 | Type Ia + Type II precursor candidate": "#77670E",
    "E18.5 | Immature Type II SGN": "#B15A4E",
    "E18.5 | Immature Type Ia SGN": "#46AB50",
    "E18.5 | Immature Type Ib SGN": "#5351B5",
    "E18.5 | Immature Type Ic SGN": "#00AB7E",
    "E18.5 | Undifferentiated SGN": "#B13843",
    "P1 | Type Ia": "#E58E85",
    "P1 | Type Ib": "#097739",
    "P1 | Type Ic": "#784967",
    "P1 | Type II": "#5C7634",
    "P7 | Type Ia": "#009BE7",
    "P7 | Type Ib": "#E77640",
    "P7 | Type Ic": "#5DAB2B",
    "P7 | Type II": "#D285E6",
    "P28 | Type Ia": "#776746",
    "P28 | Type Ib": "#62AB8A",
    "P28 | Type Ic": "#8C69B1",
    "P28 | Type II": "#AAA147",
}

#2. =========== Plot-only parameters
UMAP_AXIS_SIZE = 5.6                 # Source PDF axes size, inches
UMAP_FINAL_SCALE = {"Embryonic": 1.0, "Postnatal": 1.0, "Whole_Period": 725 / 475}
                               # Reference screenshot axes: about 725 px versus 475 px
UMAP_POINT_SIZE = {"Embryonic": 1.35, "Postnatal": 10.0, "Whole_Period": 1.35}
UMAP_POINT_ALPHA = 0.82              # Shared by scatter and legend markers
UMAP_AXIS_LINEWIDTH = 1.1            # Target width after placement, pt
UMAP_TICK_LINEWIDTH = 0.8
UMAP_TICK_LENGTH = 3.0
UMAP_TICK_FONTSIZE = 7.0
UMAP_AXIS_FONTSIZE = 9.0
UMAP_TITLE_FONTSIZE = 10.0
UMAP_LABEL_FONTSIZE = 6.75
UMAP_LEGEND_FONTSIZE = 7.5
UMAP_LEGEND_MARKER_SIZE = 5.0
UMAP_TICK_STEP = {"Embryonic": 5, "Postnatal": 2, "Whole_Period": 5}
UMAP_LABEL_WRAP = 25
UMAP_LEGEND_WRAP = 47
UMAP_LABEL_OFFSETS_PT = {}           # {"dataset:Cell or Stage": {category: [dx, dy]}}; Cell offsets are source-page pt.
UMAP_CONNECTOR_LINEWIDTH = 0.42
UMAP_CONNECTOR_ALPHA = 0.70
UMAP_CONNECTOR_DISTANCE = 12.0       # Point-to-text-edge distance, target pt

# Cell captions and legends share the approved DotPlot display names.
UMAP_DISPLAY_REPLACEMENTS = {
    " | ": " ", " (not SGN-specific)": "",
    "Type Ia + Type II precursor candidate": "Type Ia/II precursor candidate",
    "Type Ib + Type Ic precursor candidate": "Type Ib/Ic precursor candidate",
    "Immature Type Ia SGN": "Immature Type Ia", "Immature Type Ib SGN": "Immature Type Ib",
    "Immature Type Ic SGN": "Immature Type Ic", "Immature Type II SGN": "Immature Type II",
}
UMAP_CELL_LABEL_BREAKS = {
    "Undifferentiated SGN": "Undifferentiated\nSGN",
    "Type Ia/II precursor candidate": "Type Ia/II\nprecursor candidate",
    "Type Ib/Ic precursor candidate": "Type Ib/Ic\nprecursor candidate",
}
UMAP_CELL_CLEARANCE_PT = 2.0         # Additional source-page clearance around every cell glyph.
UMAP_LABEL_GAP_PT = 2.0
UMAP_GRID_STEP_PT = 3.0
UMAP_CELL_BIN_PT = 0.5
UMAP_ORDER_TRIALS = 16
UMAP_MAX_ORDER_TRIALS = 48
UMAP_LEADER_MAX_WEIGHT = 4.0
UMAP_LABEL_RANDOM_SEED = 42


assert set(UMAP_CELL_ANNOTATION_COLORS) == set(umap_metadata["cell_palette"])
UMAP_STAGE_COLORS = dict(umap_metadata["stage_palette"])

## 4. UMAP axes and deterministic label layout
The automatic layout is retained for future deliberate layout changes. The default path uses the final saved positions and links each Cell label to a verified, category-owned representative cell.

In [ ]:
#4. =========== UMAP axes and label helpers
def umap_axis_limits(xy, dataset):
    lower, upper = xy.min(axis=0), xy.max(axis=0)
    if dataset == "Whole_Period":
        center = (lower + upper) / 2
        span = float(np.ptp(xy, axis=0).max()) * 1.12
        return np.column_stack((center - span / 2, center + span / 2))
    padding = np.maximum((upper - lower) * (0.08 if dataset == "Postnatal" else 0.055), 0.1 if dataset == "Postnatal" else 0.3)
    return np.column_stack((lower - padding, upper + padding))


def umap_make_figure(dataset, kind, xy, keys, palette, category_order):
    scale = UMAP_FINAL_SCALE[dataset]
    left, bottom, right, top = 0.65 / scale, 0.62 / scale, 3.9 / scale, 0.48 / scale
    width, height = left + UMAP_AXIS_SIZE + right, bottom + UMAP_AXIS_SIZE + top
    fig = plt.figure(figsize=(width, height), dpi=144, facecolor="white")
    ax = fig.add_axes([left / width, bottom / height, UMAP_AXIS_SIZE / width, UMAP_AXIS_SIZE / height])
    order = np.random.default_rng(UMAP_RANDOM_SEED).permutation(len(xy))
    ax.scatter(*xy[order].T, c=[palette[key] for key in keys[order]],
               s=UMAP_POINT_SIZE[dataset], alpha=UMAP_POINT_ALPHA, linewidths=0, edgecolors="none")
    limits = umap_axis_limits(xy, dataset)
    ax.set(xlim=limits[0], ylim=limits[1])
    ax.set_box_aspect(1)
    ax.set_xlabel("UMAP1", fontsize=UMAP_AXIS_FONTSIZE / scale, labelpad=5 / scale)
    ax.set_ylabel("UMAP2", fontsize=UMAP_AXIS_FONTSIZE / scale, labelpad=5 / scale)
    title = dataset.replace("_", " ").replace("Whole Period", "Whole period")
    ax.set_title(title + (": Cell annotation" if kind == "Cell" else ": Stage distribution"),
                 fontsize=UMAP_TITLE_FONTSIZE / scale, pad=10 / scale)
    for axis in (ax.xaxis, ax.yaxis):
        axis.set_major_locator(MultipleLocator(UMAP_TICK_STEP[dataset]))
        axis.set_major_formatter(FormatStrFormatter("%d"))
        axis.set_minor_locator(matplotlib.ticker.NullLocator())
    ax.tick_params(labelsize=UMAP_TICK_FONTSIZE / scale, pad=3 / scale,
                   length=UMAP_TICK_LENGTH / scale, width=UMAP_TICK_LINEWIDTH / scale,
                   colors="black", direction="out")
    ax.spines[["top", "right"]].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set(color="black", linewidth=UMAP_AXIS_LINEWIDTH / scale)
    handles = []
    for key in category_order:
        caption = umap_display_annotation(key) if kind == "Cell" else key.replace("_", " ")
        label = f"{caption} (n={np.count_nonzero(keys == key):,})"
        if kind != "Cell":
            label = textwrap.fill(label, width=UMAP_LEGEND_WRAP, break_long_words=False, break_on_hyphens=False)
        handles.append(Line2D([], [], linestyle="", marker="o", markerfacecolor=palette[key],
                              markeredgecolor="none", markersize=UMAP_LEGEND_MARKER_SIZE / scale,
                              alpha=UMAP_POINT_ALPHA, label=label))
    legend = ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.025, 0.5),
                       frameon=False, fontsize=UMAP_LEGEND_FONTSIZE / scale,
                       handletextpad=0.35, labelspacing=0.50, borderaxespad=0)
    fig.canvas.draw()
    legend_box = legend.get_window_extent(fig.canvas.get_renderer())
    width = legend_box.x1 / fig.dpi + 0.18 / scale
    fig.set_size_inches(width, height)
    ax.set_position([left / width, bottom / height, UMAP_AXIS_SIZE / width, UMAP_AXIS_SIZE / height])
    return fig, ax


def umap_display_annotation(key):
    for old,new in UMAP_DISPLAY_REPLACEMENTS.items():
        key = key.replace(old,new)
    return key


def umap_cell_caption(key,dataset):
    stage,name=key.split(" | ",1)
    name=umap_display_annotation(name)
    semantic_breaks=UMAP_CELL_LABEL_BREAKS
    if name in semantic_breaks:
        name=semantic_breaks[name]
    else:
        name=textwrap.fill(name,UMAP_LABEL_WRAP,break_long_words=False,break_on_hyphens=False)
    return stage+"\n"+name


def umap__cell_boxes_overlap(boxes, box, pad=0):
    return ((boxes[:,0] < box[2]+pad) & (boxes[:,2] > box[0]-pad)
            & (boxes[:,1] < box[3]+pad) & (boxes[:,3] > box[1]-pad))


def umap__cell_segment_boxes(start,end,boxes,pad=0):
    """Vectorized segment-versus-rectangle intersection using interval clipping."""
    starts = np.broadcast_to(start,(len(boxes),2))
    ends = np.broadcast_to(end,(len(boxes),2))
    delta = ends-starts
    low = np.zeros(len(boxes)); high = np.ones(len(boxes))
    for axis in range(2):
        d = delta[:,axis]
        parallel = np.abs(d) < 1e-12
        outside = parallel & ((starts[:,axis] < boxes[:,axis]-pad) | (starts[:,axis] > boxes[:,axis+2]+pad))
        safe = np.where(parallel,1,d)
        t1 = (boxes[:,axis]-pad-starts[:,axis])/safe
        t2 = (boxes[:,axis+2]+pad-starts[:,axis])/safe
        near = np.where(parallel,-np.inf,np.minimum(t1,t2))
        far = np.where(parallel,np.inf,np.maximum(t1,t2))
        low = np.maximum(low,near); high = np.minimum(high,far)
        high[outside] = -1
    return high >= low


def umap__cell_point_rect_count(points, boxes, pad):
    result = np.zeros(len(boxes),dtype=int)
    for i,box in enumerate(boxes):
        result[i] = np.count_nonzero((points[:,0] >= box[0]-pad) & (points[:,0] <= box[2]+pad)
                     & (points[:,1] >= box[1]-pad) & (points[:,1] <= box[3]+pad))
    return result

In [ ]:
#5. =========== Optional label placement search
def umap__cell_free_candidates(points, bounds, widths, heights, glyph_radius):
    origin = bounds[:2]-2
    shape = np.ceil((bounds[2:]-origin+3)/UMAP_CELL_BIN_PT).astype(int)+1
    occupancy = np.zeros((shape[1],shape[0]),dtype=np.int32)
    indices = np.floor((points-origin)/UMAP_CELL_BIN_PT).astype(int)
    np.add.at(occupancy,(indices[:,1],indices[:,0]),1)
    integral = np.pad(occupancy,((1,0),(1,0))).cumsum(0).cumsum(1)
    xx,yy = np.meshgrid(np.arange(bounds[0],bounds[2]+.01,UMAP_GRID_STEP_PT),
                        np.arange(bounds[1],bounds[3]+.01,UMAP_GRID_STEP_PT))
    grid = np.column_stack((xx.ravel(),yy.ravel()))
    candidates=[]
    for width,height in zip(widths,heights):
        boxes = np.column_stack((grid[:,0]-width/2,grid[:,1]-height/2,
                                 grid[:,0]+width/2,grid[:,1]+height/2))
        inside = (boxes[:,0] >= bounds[0]+1) & (boxes[:,1] >= bounds[1]+1) & (boxes[:,2] <= bounds[2]-1) & (boxes[:,3] <= bounds[3]-1)
        boxes=boxes[inside]
        pad = glyph_radius+UMAP_CELL_CLEARANCE_PT
        lower=np.maximum(np.floor((boxes[:,:2]-pad-origin)/UMAP_CELL_BIN_PT).astype(int)-1,0)
        upper=np.minimum(np.floor((boxes[:,2:]+pad-origin)/UMAP_CELL_BIN_PT).astype(int)+2,shape)
        count = integral[upper[:,1],upper[:,0]]-integral[lower[:,1],upper[:,0]]-integral[upper[:,1],lower[:,0]]+integral[lower[:,1],lower[:,0]]
        candidates.append(boxes[count==0])
    return candidates


def umap__cell_make_orders(widths, heights, candidates, anchors):
    area=widths*heights
    orders=[np.argsort(-area,kind="stable"),np.argsort([len(c) for c in candidates],kind="stable"),
            np.argsort(anchors[:,1]),np.argsort(-anchors[:,1]),np.argsort(anchors[:,0]),np.argsort(-anchors[:,0])]
    rng=np.random.default_rng(UMAP_LABEL_RANDOM_SEED)
    for _ in range(UMAP_ORDER_TRIALS-len(orders)):
        orders.append(np.argsort(-area*rng.uniform(.65,1.35,len(area))))
    return orders


def umap__cell_assign_candidates(candidates, anchors, order):
    accepted=[]; leaders=[]; result={}
    total=0
    for index in order:
        boxes=candidates[index]
        if not len(boxes): return None
        valid=np.ones(len(boxes),dtype=bool)
        for other in accepted:
            valid &= ~umap__cell_boxes_overlap(boxes,other,UMAP_LABEL_GAP_PT)
        for a,b in leaders:
            valid &= ~umap__cell_segment_boxes(a,b,boxes,pad=.7)
        boxes=boxes[valid]
        if not len(boxes): return None
        anchor=anchors[index]
        starts=np.clip(anchor,boxes[:,:2],boxes[:,2:])
        valid=np.ones(len(boxes),dtype=bool)
        for other in accepted:
            valid &= ~umap__cell_segment_boxes(starts,anchor,np.broadcast_to(other,(len(boxes),4)),pad=.7)
        boxes,starts=boxes[valid],starts[valid]
        if not len(boxes): return None
        distances=np.linalg.norm(starts-anchor,axis=1)
        # Prefer nearby empty space and use distance to break ties.
        costs=distances+.025*np.linalg.norm((boxes[:,:2]+boxes[:,2:])/2-anchor,axis=1)
        j=np.argmin(costs)
        box=boxes[j];start=starts[j]
        accepted.append(box);leaders.append((start,anchor))
        result[int(index)]=(box.copy(),start.copy(),anchor.copy())
        total+=float(costs[j])
    return total,result


def umap_place_cell_labels(fig,ax,dataset,xy,keys,category_order):
    scale=UMAP_FINAL_SCALE[dataset]
    artists=[];anchors_data=[];anchor_indices=[]
    for key in category_order:
        points=xy[keys==key];center=np.median(points,axis=0)
        local_index=int(np.argmin(np.sum((points-center)**2,axis=1)))
        anchor=points[local_index]
        anchor_indices.append(int(np.flatnonzero(keys==key)[local_index]))
        caption=umap_cell_caption(key,dataset)
        artists.append(ax.text(*anchor,caption,fontsize=UMAP_LABEL_FONTSIZE/scale,
                               ha="center",va="center",color="black",linespacing=1.0,zorder=5))
        anchors_data.append(anchor)
    fig.canvas.draw();renderer=fig.canvas.get_renderer();unit=72/fig.dpi
    bbox=ax.get_window_extent(renderer)
    bounds=np.array([bbox.x0,bbox.y0,bbox.x1,bbox.y1])*unit
    text_boxes=[a.get_window_extent(renderer) for a in artists]
    widths=np.array([box.width for box in text_boxes])*unit
    heights=np.array([box.height for box in text_boxes])*unit
    points=ax.transData.transform(xy)*unit
    anchors=ax.transData.transform(anchors_data)*unit
    glyph_radius=math.sqrt(UMAP_POINT_SIZE[dataset])/2
    candidates=umap__cell_free_candidates(points,bounds,widths,heights,glyph_radius)
    overrides=UMAP_LABEL_OFFSETS_PT.get(dataset+":Cell",{})
    assert set(overrides) <= set(category_order), "Unknown Cell label override."
    fixed=[]
    for i,key in enumerate(category_order):
        if key not in overrides:
            continue
        offset=np.asarray(overrides[key],dtype=float)
        assert offset.shape==(2,) and np.isfinite(offset).all(), "Invalid Cell label offset."
        origin=artists[i].get_transform().transform(artists[i].get_position())*unit
        old_box=text_boxes[i]
        center=np.array([(old_box.x0+old_box.x1)/2,(old_box.y0+old_box.y1)/2])*unit
        target=anchors[i]+offset+(center-origin)
        box=np.r_[target-[widths[i]/2,heights[i]/2],target+[widths[i]/2,heights[i]/2]]
        assert np.all(box[:2]>=bounds[:2]+1) and np.all(box[2:]<=bounds[2:]-1), f"Cell override outside axes: {key}"
        assert umap__cell_point_rect_count(points,np.asarray([box]),glyph_radius+UMAP_CELL_CLEARANCE_PT)[0]==0, f"Cell override overlaps cell glyphs: {key}"
        candidates[i]=np.asarray([box]);fixed.append(i)
    best=None;best_score=float("inf")
    orders=umap__cell_make_orders(widths,heights,candidates,anchors)
    seen_orders={tuple(map(int,order)) for order in orders}
    for trial,order in enumerate(orders):
        if trial >= UMAP_MAX_ORDER_TRIALS:
            break
        order=np.r_[fixed,[i for i in order if i not in fixed]].astype(int)
        assignment=umap__cell_assign_candidates(candidates,anchors,order)
        if assignment is None:
            continue
        lengths={i:float(np.linalg.norm(v[1]-v[2])) for i,v in assignment[1].items()}
        score=assignment[0]+UMAP_LEADER_MAX_WEIGHT*max(lengths.values())
        if score < best_score:
            best,best_score=assignment,score
        # Place labels with the longest leaders first in the next pass.
        # Shorten long leaders without moving cells or changing the font size.
        for index in sorted(lengths,key=lengths.get,reverse=True)[:2]:
            fresh=np.r_[index,[i for i in order if i!=index]]
            key=tuple(map(int,fresh))
            if key not in seen_orders:
                seen_orders.add(key);orders.append(fresh)
    if best is None:
        raise ValueError(f"No valid Cell label layout: {dataset}; check manual offsets.")
    _,placed=best;positions={}
    for i,artist in enumerate(artists):
        box,start,end=placed[i]
        target_center=(box[:2]+box[2:])/2
        old_origin=artist.get_transform().transform(artist.get_position())*unit
        old_box=text_boxes[i]
        old_center=(np.array([old_box.x0,old_box.y0])+np.array([old_box.x1,old_box.y1]))/2*unit
        data=ax.transData.inverted().transform((target_center-(old_center-old_origin))/unit)
        artist.set_position(data);positions[category_order[i]]=data.tolist()
        line=ax.transData.inverted().transform(np.vstack((start,end))/unit)
        ax.plot(*line.T,color="0.32",lw=UMAP_CONNECTOR_LINEWIDTH/scale,
                alpha=UMAP_CONNECTOR_ALPHA,zorder=3)
    fig.canvas.draw();renderer=fig.canvas.get_renderer()
    boxes=np.asarray([[b.x0,b.y0,b.x1,b.y1] for b in (a.get_window_extent(renderer) for a in artists)])*unit
    collisions=umap__cell_point_rect_count(points,boxes,glyph_radius+UMAP_CELL_CLEARANCE_PT)
    label_collisions=sum(umap__cell_boxes_overlap(boxes[i+1:],b,UMAP_LABEL_GAP_PT-.001).sum() for i,b in enumerate(boxes))
    line_collisions=0
    lengths=[]
    for i in range(len(artists)):
        _,a,b=placed[i]
        line_collisions+=int(umap__cell_segment_boxes(a,b,np.delete(boxes,i,axis=0),pad=.6).sum())
        lengths.append(float(np.linalg.norm(a-b)))
    contained=bool(np.all(boxes[:,:2]>=bounds[:2]-.01) and np.all(boxes[:,2:]<=bounds[2:]+.01))
    assert not collisions.any() and not label_collisions and not line_collisions and contained
    fig._cell_layout_metrics=dict(dataset=dataset,feasible=True,labels=len(artists),
            cell_glyph_padding_pt=UMAP_CELL_CLEARANCE_PT,cell_intersections=int(collisions.sum()),
            label_intersections=int(label_collisions),line_label_intersections=line_collisions,
            contained_in_fixed_axes=contained,max_leader_pt=max(lengths),median_leader_pt=float(np.median(lengths)),
            total_leader_pt=sum(lengths),manual_overrides=len(fixed))
    fig._cell_label_anchors={key:{"cell_index":int(index),"xy":list(map(float,anchor))}
                             for key,index,anchor in zip(category_order,anchor_indices,anchors_data)}
    return positions


def umap_place_labels(fig, ax, dataset, kind, xy, keys, category_order):
    if kind == "Cell":
        return umap_place_cell_labels(fig,ax,dataset,xy,keys,category_order)
    scale = UMAP_FINAL_SCALE[dataset]
    texts, anchors = [], []
    for key in category_order:
        points = xy[keys == key]
        center = np.median(points, axis=0)
        anchor = points[np.argmin(np.sum((points - center) ** 2, axis=1))]
        caption = key
        if kind == "Cell":
            stage, name = key.split(" | ", 1)
            caption = stage + "\n" + textwrap.fill(name, UMAP_LABEL_WRAP, break_long_words=False, break_on_hyphens=False)
        texts.append(ax.text(*anchor, caption, fontsize=UMAP_LABEL_FONTSIZE / scale,
                             ha="center", va="center", color="black", linespacing=1.0, zorder=5))
        anchors.append(anchor)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    bounds = ax.get_window_extent(renderer).padded(-2)
    pixels = ax.transData.transform(xy)
    anchor_pixels = ax.transData.transform(anchors)
    boxes = [text.get_window_extent(renderer) for text in texts]
    sizes = np.array([[box.width + 3 / scale, box.height + 3 / scale] for box in boxes])
    accepted, positions = [], {}
    overrides = UMAP_LABEL_OFFSETS_PT.get(dataset + ":" + kind, {})
    for i in sorted(range(len(texts)), key=lambda i: -np.prod(sizes[i])):
        w, h = sizes[i]
        anchor = anchor_pixels[i]
        options = []
        for radius in (0, 6, 12, 18, 24, 32, 42, 55, 70, 90, 115):
            angles = [0] if radius == 0 else np.linspace(0, 2 * np.pi, 32, endpoint=False)
            for angle in angles:
                offset = np.array([np.cos(angle), np.sin(angle)]) * radius / scale * fig.dpi / 72
                if category_order[i] in overrides:
                    offset = np.asarray(overrides[category_order[i]]) / scale * fig.dpi / 72
                pos = anchor + offset
                box = Bbox.from_bounds(pos[0] - w / 2, pos[1] - h / 2, w, h)
                if not (bounds.contains(*box.p0) and bounds.contains(*box.p1)):
                    continue
                if any(box.overlaps(other) for other in accepted):
                    continue
                covered = np.count_nonzero((pixels[:, 0] >= box.x0) & (pixels[:, 0] <= box.x1)
                                          & (pixels[:, 1] >= box.y0) & (pixels[:, 1] <= box.y1))
                score = np.linalg.norm(offset) * 72 / fig.dpi * scale + 0.18 * covered
                options.append((score, pos, box))
            if options and radius >= 18:
                break
        assert options, f"No label position: {dataset} {kind} {category_order[i]}"
        _, pos, box = min(options, key=lambda item: item[0])
        texts[i].set_position(ax.transData.inverted().transform(pos))
        positions[category_order[i]] = list(map(float, texts[i].get_position()))
        accepted.append(box)
    fig.canvas.draw()
    boxes = [text.get_window_extent(renderer) for text in texts]
    for i, (text, box) in enumerate(zip(texts, boxes)):
        point = anchor_pixels[i]
        edge = np.clip(point, box.p0, box.p1)
        distance = np.linalg.norm(point - edge) * 72 / fig.dpi * scale
        if distance > UMAP_CONNECTOR_DISTANCE:
            segment = ax.transData.inverted().transform([edge, point])
            ax.plot(*segment.T, color="0.32", lw=UMAP_CONNECTOR_LINEWIDTH / scale,
                    alpha=UMAP_CONNECTOR_ALPHA, zorder=3)
    assert not any(a.overlaps(b) for i, a in enumerate(boxes) for b in boxes[i + 1:])
    return positions

In [ ]:
#6. =========== Reuse the final saved label positions
def umap_place_saved_labels(fig, ax, dataset, kind, xy, keys, category_order):
    scale = UMAP_FINAL_SCALE[dataset]
    positions = umap_metadata["label_positions_data"][dataset+":"+kind]
    assert set(positions) == set(category_order)
    artists, anchors = [], []
    for key in category_order:
        if kind == "Cell":
            item = umap_metadata["cell_label_anchors"][dataset][key]
            index = item["cell_index"]
            assert keys[index] == key
            assert str(umap_arrays[dataset+"_cell_ids"][index]) == item["cell_id"]
            assert np.array_equal(xy[index],item["xy"])
            anchor = xy[index]
            caption = umap_cell_caption(key,dataset)
        else:
            points = xy[keys == key]
            center = np.median(points,axis=0)
            anchor = points[np.argmin(np.sum((points-center)**2,axis=1))]
            caption = key
        artist = ax.text(*positions[key],caption,fontsize=UMAP_LABEL_FONTSIZE/scale,
                         ha="center",va="center",color="black",linespacing=1.0,zorder=5)
        artists.append(artist)
        anchors.append(anchor)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    unit = 72/fig.dpi
    boxes = [artist.get_window_extent(renderer) for artist in artists]
    for anchor,box in zip(anchors,boxes):
        point = ax.transData.transform(anchor)
        edge = np.clip(point,box.p0,box.p1)
        distance = np.linalg.norm(point-edge)*unit*scale
        if kind == "Cell" or distance > UMAP_CONNECTOR_DISTANCE:
            segment = ax.transData.inverted().transform([edge,point])
            ax.plot(*segment.T,color="0.32",lw=UMAP_CONNECTOR_LINEWIDTH/scale,
                    alpha=UMAP_CONNECTOR_ALPHA,zorder=3)
    assert not any(a.overlaps(b) for i,a in enumerate(boxes) for b in boxes[i+1:])
    if kind == "Cell":
        glyph_radius = math.sqrt(UMAP_POINT_SIZE[dataset])/2
        glyph_points = ax.transData.transform(xy)*unit
        rects = np.asarray([[b.x0,b.y0,b.x1,b.y1] for b in boxes])*unit
        assert not umap__cell_point_rect_count(glyph_points,rects,glyph_radius+UMAP_CELL_CLEARANCE_PT).any()
    return positions

## 7. Editable PDF exporter
ReportLab writes independent vector circles, lines and semantic text objects in three sibling clipping groups. Illustrator supplies real multiline text frames in its native editable payload. The exporter reopens each file in Illustrator to check frame contents, fonts and clipping depth; it restores the previously active document.

The two embryonic PDFs were subsequently saved by Illustrator. To reproduce that appearance exactly, their newly generated native presentation is normalized without changing its Adobe kerning, rounded coordinates or opacity. Each native Form contains one paint operation; its group opacity is folded into that operation before inlining. WinAnsi Unicode mappings are added and complete text frames are retained. These two pages need no clipping; the other five retain the three-group presentation. The existing final PDFs are never read as drawing input.

In [ ]:
#7. =========== Native editable text frames and Adobe presentation normalization
from collections import Counter
from decimal import Decimal
from pypdf.generic import (DecodedStreamObject, DictionaryObject, FloatObject,
                          NumberObject, TextStringObject, ByteStringObject)

def _number(value):
    return FloatObject(str(value))


def _winansi_unicode(writer, font):
    """Add an explicit, complete Unicode map to the native embedded Arial font."""
    if font.get("/ToUnicode"):
        return
    assert font.get("/Encoding") == "/WinAnsiEncoding", font.get("/Encoding")
    assert font.get("/Subtype") == "/TrueType"
    assert "Arial" in str(font.get("/BaseFont"))
    assert "/FontFile2" in font["/FontDescriptor"]
    pairs = []
    for code in range(256):
        try:
            char = bytes([code]).decode("cp1252")
        except UnicodeDecodeError:
            continue
        pairs.append(f"<{code:02X}> <{char.encode('utf-16-be').hex().upper()}>")
    chunks = [pairs[i:i+100] for i in range(0, len(pairs), 100)]
    body = ["/CIDInit /ProcSet findresource begin", "12 dict begin", "begincmap",
            "/CIDSystemInfo << /Registry (Adobe) /Ordering (UCS) /Supplement 0 >> def",
            "/CMapName /ArialWinAnsiUCS def", "/CMapType 2 def",
            "1 begincodespacerange", "<00> <FF>", "endcodespacerange"]
    for chunk in chunks:
        body += [f"{len(chunk)} beginbfchar", *chunk, "endbfchar"]
    body += ["endcmap", "CMapName currentdict /CMap defineresource pop", "end", "end"]
    stream = DecodedStreamObject()
    stream.set_data(("\n".join(body)+"\n").encode("ascii"))
    font[NameObject("/ToUnicode")] = writer._add_object(stream)


def _shown_text(operands, operator):
    pieces = operands[0] if operator == b"TJ" else operands
    result = []
    for value in pieces:
        if isinstance(value, TextStringObject):
            result.append(str(value))
        elif isinstance(value, ByteStringObject):
            result.append(bytes(value).decode("cp1252"))
    return "".join(result)


def _semantic_text(operations, specs):
    """Resolve native Td positioning, preserving every native TJ and word space."""
    lines = []
    matrix = tuple(map(Decimal, (1, 0, 0, 1, 0, 0)))
    state = {b"Tc": [NumberObject(0)], b"Tw": [NumberObject(0)],
             b"Tz": [NumberObject(100)], b"TL": [NumberObject(0)],
             b"Tr": [NumberObject(0)], b"Ts": [NumberObject(0)],
             b"rg": [NumberObject(0)]*3}
    moved = False
    for operands, operator in operations:
        if operator in (b"BT", b"ET"):
            continue
        if operator == b"Tm":
            matrix = tuple(Decimal(str(value)) for value in operands)
            moved = True
        elif operator in (b"Td", b"TD"):
            x, y = map(lambda value: Decimal(str(value)), operands)
            a,b,c,d,e,f = matrix
            matrix = (a,b,c,d,e+a*x+c*y,f+b*x+d*y)
            if operator == b"TD":
                state[b"TL"] = [_number(-y)]
            moved = True
        elif operator in (b"Tf",b"Tc",b"Tw",b"Tz",b"TL",b"Tr",b"Ts",b"rg"):
            state[operator] = operands
        elif operator in (b"Tj",b"TJ"):
            assert moved, "Consecutive text-show operators need explicit advance handling."
            lines.append(dict(text=_shown_text(operands,operator),matrix=matrix,
                              state=dict(state),show=(operands,operator)))
            moved = False
        else:
            raise AssertionError(f"Unexpected native text operator: {operator!r}")
    remaining = []
    for item in specs:
        text = item.get("text",item.get("contents"))
        remaining.append(text.replace("\r\n","\n").replace("\r","\n").split("\n"))
    output = []
    index = 0
    groups = 0
    while index < len(lines):
        matches = [i for i,expected in enumerate(remaining)
                   if [line["text"] for line in lines[index:index+len(expected)]] == expected]
        assert matches, (index,[line["text"] for line in lines[index:index+4]])
        selected = max(matches,key=lambda i:len(remaining[i]))
        count = len(remaining.pop(selected))
        output.append(([],b"BT"))
        for line in lines[index:index+count]:
            for operator in (b"rg",b"Tf",b"Tc",b"Tw",b"Tz",b"TL",b"Tr",b"Ts"):
                output.append((line["state"][operator],operator))
            output.append((list(map(_number,line["matrix"])),b"Tm"))
            output.append(line["show"])
        output.append(([],b"ET"))
        index += count
        groups += 1
    assert not remaining, remaining
    return output,groups


def normalize_native_pdf(source, destination, specs):
    """Keep native Illustrator payload; inline single-path Forms and semantic text.

    The resulting page contains zero clips, Forms, images, transparency groups,
    or soft masks. Plain fill/stroke alpha and Adobe font kerning are preserved.
    """
    source,destination = Path(source),Path(destination)
    reader = PdfReader(source)
    assert len(reader.pages) == 1
    writer = PdfWriter()
    writer.clone_document_from_reader(reader)
    page = writer.pages[0]
    resources = DictionaryObject()
    fonts,graphics = DictionaryObject(),DictionaryObject()
    resources[NameObject("/Font")] = fonts
    resources[NameObject("/ExtGState")] = graphics
    resource_cache = {}
    paint_operators = {b"f",b"F",b"f*",b"S",b"s",b"B",b"B*",b"b",b"b*",b"Do",b"Tj",b"TJ"}
    forms_inlined = 0

    def get_resource(category,name,local,multiplier=(1.0,1.0)):
        obj = local[category][name].get_object()
        if category == "/Font":
            key = (category,id(obj))
        else:
            key = (category,id(obj),multiplier)
        if key in resource_cache:
            return resource_cache[key]
        newname = NameObject(f"/N{len(resource_cache):05d}")
        copy = (obj.clone(writer) if category == "/Font" else
                DictionaryObject({key:value.clone(writer) for key,value in obj.items()}))
        if category == "/Font":
            _winansi_unicode(writer,copy)
            fonts[newname] = writer._add_object(copy)
        else:
            assert copy.get("/SMask","/None") == "/None"
            assert copy.get("/BM","/Normal") == "/Normal"
            copy.pop("/SMask",None)
            copy[NameObject("/ca")] = _number(float(copy.get("/ca",1))*multiplier[0])
            copy[NameObject("/CA")] = _number(float(copy.get("/CA",1))*multiplier[1])
            graphics[newname] = writer._add_object(copy)
        resource_cache[key] = newname
        return newname

    def flatten(operations,local,multiplier=(1.0,1.0),initial=(1.0,1.0)):
        nonlocal forms_inlined
        output,stack = [],[]
        alpha = initial
        for operands,operator in operations:
            if operator == b"q":
                stack.append(alpha)
            elif operator == b"Q":
                alpha = stack.pop()
            elif operator in (b"W",b"W*",b"BDC",b"BMC",b"EMC",b"MP",b"DP"):
                continue
            elif operator == b"gs":
                gs = local["/ExtGState"][operands[0]].get_object()
                alpha = (float(gs.get("/ca",alpha[0])),float(gs.get("/CA",alpha[1])))
                operands = [get_resource("/ExtGState",operands[0],local,multiplier)]
            elif operator == b"Tf":
                operands = [get_resource("/Font",operands[0],local),operands[1]]
            elif operator == b"Do":
                form = local["/XObject"][operands[0]].get_object()
                assert form["/Subtype"] == "/Form"
                group = form.get("/Group",{})
                assert not getattr(group.get("/I",False),"value",False)
                assert not getattr(group.get("/K",False),"value",False)
                inner = ContentStream(form,reader).operations
                assert sum(op in paint_operators for _,op in inner) == 1
                output.append(([],b"q"))
                if "/Matrix" in form:
                    output.append((list(form["/Matrix"]),b"cm"))
                factor = (multiplier[0]*alpha[0],multiplier[1]*alpha[1]) if group else multiplier
                output.extend(flatten(inner,form.get("/Resources",local),factor,initial=alpha))
                output.append(([],b"Q"))
                forms_inlined += 1
                continue
            output.append((operands,operator))
        assert not stack
        return output

    native = ContentStream(reader.pages[0]["/Contents"],reader).operations
    operations = flatten(native,reader.pages[0]["/Resources"])
    begins = [i for i,(_,op) in enumerate(operations) if op == b"BT"]
    ends = [i for i,(_,op) in enumerate(operations) if op == b"ET"]
    assert len(begins) == len(ends) == 1
    begin,end = begins[0],ends[0]
    text,frame_count = _semantic_text(operations[begin:end+1],specs)
    operations[begin:end+1] = text
    content = ContentStream(None,writer)
    content.operations = operations
    stream = DecodedStreamObject()
    stream.set_data(content.get_data())
    page[NameObject("/Contents")] = writer._add_object(stream.flate_encode())
    page[NameObject("/Resources")] = resources
    page.pop("/Group",None)
    writer.write(destination)
    assert page.get("/PieceInfo"), "Native Illustrator payload was lost."
    counts = Counter(op for _,op in operations)
    assert counts[b"BT"] == counts[b"ET"] == len(specs)
    assert not any(counts[x] for x in (b"Do",b"W",b"W*"))
    return dict(forms_inlined=forms_inlined,semantic_text_frames=frame_count,
                clips=0,fonts=len(fonts),extgstates=len(graphics),path=str(destination))

def illustrator_script(source):
    # Set the temporary working directory or file path.
    script_path = ROOT
    script_path.write_text(source, encoding="utf-8")
    path_literal = str(script_path).replace("'", "''")
    command = ("$ErrorActionPreference='Stop'; "
               "$illustratorApp=New-Object -ComObject Illustrator.Application; "
               f"$source=[System.IO.File]::ReadAllText('{path_literal}'); "
               "$illustratorApp.DoJavaScript($source)")
    encoded = base64.b64encode(command.encode("utf-16le")).decode("ascii")
    result = subprocess.run(["powershell.exe", "-NoProfile", "-NonInteractive", "-EncodedCommand", encoded],
                            capture_output=True, text=True, timeout=180, check=True)
    if "PASS" not in result.stdout:
        raise RuntimeError(result.stdout + result.stderr)
    return result.stdout.strip()


def native_text_pdf(base_path, destination, specs):
    # Set the output file path.
    native_path = ROOT
    # Set the output file path.
    merged_path = ROOT
    values = {"base": base_path.as_posix(), "native": native_path.as_posix(),
              "verify": merged_path.as_posix(), "specs": specs}
    prefix = "var job=" + json.dumps(values, ensure_ascii=True) + ";\n"
    check_function = r"""
function verifyText(d) {
    if (d.textFrames.length !== job.specs.length) throw Error('Wrong text-frame count: '+d.textFrames.length);
    var seen = {};
    for (var i=0;i<d.textFrames.length;i++) {
        var t=d.textFrames[i], key=t.name;
        if (seen[key]) throw Error('Duplicate text frame: '+key);
        if (t.textRange.characterAttributes.textFont.name !== 'ArialMT') throw Error('Unexpected font: '+key);
        seen[key]=t.contents;
    }
    for (var i=0;i<job.specs.length;i++) {
        var s=job.specs[i];
        if (seen[s.name] !== s.text.replace(/\n/g,'\r')) throw Error('Fragmented or changed text: '+s.name);
    }
    var clips=0,maxdepth=0;
    for(var i=0;i<d.groupItems.length;i++) {
        var g=d.groupItems[i];if(!g.clipped)continue;
        clips++;var depth=1,parent=g.parent;
        while(parent.typename==='GroupItem'){if(parent.clipped)depth++;parent=parent.parent;}
        maxdepth=Math.max(maxdepth,depth);
    }
    if(clips!==3 || maxdepth!==1) throw Error('Unexpected native clipping structure');
}
"""
    build = r"""
(function(){
var prior=app.documents.length?app.activeDocument:null;
var level=app.userInteractionLevel, d=null;
try {
    app.userInteractionLevel=UserInteractionLevel.DONTDISPLAYALERTS;
    d=app.open(File(job.base));
    var parent=d.textFrames.length?d.textFrames[0].parent:d.layers[0];
    while(d.textFrames.length) d.textFrames[0].remove();
    var font=app.textFonts.getByName('ArialMT');
    for(var i=0;i<job.specs.length;i++) {
        var s=job.specs[i], t=d.textFrames.add();
        t.move(parent,ElementPlacement.PLACEATEND);
        t.name=s.name;
        t.contents=s.text.replace(/\n/g,'\r');
        var ca=t.textRange.characterAttributes;
        ca.textFont=font;ca.size=s.fontsize;ca.autoLeading=false;ca.leading=s.leading;
        var black=new RGBColor();black.red=black.green=black.blue=0;ca.fillColor=black;
        var pa=t.textRange.paragraphAttributes;
        pa.justification=s.align==='center'?Justification.CENTER:(s.align==='right'?Justification.RIGHT:Justification.LEFT);
        pa.spaceBefore=0;pa.spaceAfter=0;pa.hyphenation=false;
        if(s.angle) t.rotate(s.angle);
        var anchor=t.anchor;
        t.translate(s.anchor[0]-anchor[0],s.anchor[1]-anchor[1]);
        t.opacity=100;
    }
    verifyText(d);
    var opt=new PDFSaveOptions();
    opt.preserveEditability=true;opt.compatibility=PDFCompatibility.ACROBAT7;
    opt.generateThumbnails=false;opt.viewAfterSaving=false;
    d.saveAs(File(job.native),opt);
    d.close(SaveOptions.DONOTSAVECHANGES);d=null;
    d=app.open(File(job.native));verifyText(d);
    d.close(SaveOptions.DONOTSAVECHANGES);d=null;
    if(prior)prior.activate();
    return 'PASS native text frames: '+job.specs.length;
} finally {
    if(d)d.close(SaveOptions.DONOTSAVECHANGES);
    app.userInteractionLevel=level;
}
})()
"""
    illustrator_script(prefix + check_function + build)
    # Keep the three PDF groups and the embedded Unicode font.
    # Illustrator's native PieceInfo retains the same artwork with real multiline TextFrames.
    native = PdfReader(native_path)
    base = PdfReader(base_path)
    writer = PdfWriter()
    writer.clone_document_from_reader(native)
    page = writer.pages[0]
    assert "/Illustrator" in page["/PieceInfo"]
    for key in ("/Contents", "/Resources", "/MediaBox", "/CropBox", "/ArtBox", "/BleedBox", "/TrimBox"):
        if key in base.pages[0]:
            page[NameObject(key)] = base.pages[0].raw_get(key).clone(writer)
        elif key in page:
            del page[NameObject(key)]
    writer.write(merged_path)
    verify = r"""
(function(){
var prior=app.documents.length?app.activeDocument:null;
var level=app.userInteractionLevel,d=null;
try{
app.userInteractionLevel=UserInteractionLevel.DONTDISPLAYALERTS;
d=app.open(File(job.verify));verifyText(d);
var multiline=0;for(var i=0;i<d.textFrames.length;i++)if(d.textFrames[i].contents.indexOf('\r')>=0)multiline++;
d.close(SaveOptions.DONOTSAVECHANGES);d=null;if(prior)prior.activate();
return 'PASS reopened PDF: '+job.specs.length+' text frames, '+multiline+' multiline';
}finally{if(d)d.close(SaveOptions.DONOTSAVECHANGES);app.userInteractionLevel=level;}
})()
"""
    print(illustrator_script(prefix + check_function + verify))
    if destination.stem in {"Embryonic_Cell","Embryonic_Stage"}:
        normalize_native_pdf(native_path,destination,specs)
    else:
        destination.write_bytes(merged_path.read_bytes())

In [ ]:
#8. =========== UMAP vector PDF export
def umap_export_pdf(fig, ax, destination, dataset):
    TASK_TMP.mkdir(parents=True,exist_ok=True)
    os.environ["TMP"] = os.environ["TEMP"] = str(TASK_TMP)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    unit, scale = 72 / fig.dpi, UMAP_FINAL_SCALE[dataset]
    width, height = fig.get_size_inches() * 72
    assert np.allclose(ax.get_window_extent(renderer).size * unit, UMAP_AXIS_SIZE * 72)
    texts = [ax.title, ax.xaxis.label, ax.yaxis.label, *ax.texts]
    texts += [tick.label1 for tick in ax.xaxis.get_major_ticks() if ax.get_xlim()[0] <= tick.get_loc() <= ax.get_xlim()[1]]
    texts += [tick.label1 for tick in ax.yaxis.get_major_ticks() if ax.get_ylim()[0] <= tick.get_loc() <= ax.get_ylim()[1]]
    legend = ax.get_legend()
    texts += list(legend.get_texts())
    for text in texts:
        bbox = text.get_window_extent(renderer)
        assert fig.bbox.contains(*bbox.p0) and fig.bbox.contains(*bbox.p1), text.get_text()
        assert not text.get_path_effects()
    assert legend.get_window_extent(renderer).x0 > ax.get_window_extent(renderer).x1
    # Set the output file path.
    base_path = ROOT
    pdf = canvas.Canvas(str(base_path), pagesize=(width, height), pageCompression=1,
                        initialFontName="Arial", initialFontSize=7)
    pdf.setTitle(ax.get_title())

    def begin_group(name):
        pdf.saveState()
        pdf.addLiteral("% " + name)
        rectangle = pdf.beginPath()
        rectangle.rect(0, 0, width, height)
        pdf.clipPath(rectangle, stroke=0, fill=0, fillMode=1)

    def circle(point, radius, face, opacity):
        pdf.setFillColorRGB(*colors.to_rgb(face))
        pdf.setFillAlpha(float(opacity))
        pdf.circle(*map(float, point), float(radius), stroke=0, fill=1)

    def line(points, color="black", linewidth=UMAP_AXIS_LINEWIDTH / scale, opacity=1):
        pdf.setStrokeColorRGB(*colors.to_rgb(color))
        pdf.setStrokeAlpha(float(opacity))
        pdf.setLineWidth(float(linewidth))
        path = pdf.beginPath()
        path.moveTo(*points[0])
        for point in points[1:]:
            path.lineTo(*point)
        pdf.drawPath(path, stroke=1, fill=0)

    begin_group("Graphics")
    for collection in ax.collections:
        points = collection.get_offset_transform().transform(collection.get_offsets()) * unit
        faces = collection.get_facecolors()
        radius = np.sqrt(collection.get_sizes()[0]) / 2
        for i, point in enumerate(points):
            face = faces[i if len(faces) > 1 else 0]
            circle(point, radius, face[:3], face[3])
    for handle in legend.legend_handles:
        points = handle.get_transform().transform(handle.get_xydata()) * unit
        circle(points[len(points) // 2], handle.get_markersize() / 2,
               handle.get_markerfacecolor(), UMAP_POINT_ALPHA)
    pdf.restoreState()

    begin_group("Axes/Guides")
    for side in ("left", "bottom"):
        spine = ax.spines[side]
        line(spine.get_transform().transform(spine.get_path().vertices) * unit)
    for value in ax.get_xticks():
        if ax.get_xlim()[0] <= value <= ax.get_xlim()[1]:
            point = ax.transData.transform([value, ax.get_ylim()[0]]) * unit
            line([point, point + [0, -UMAP_TICK_LENGTH / scale]], linewidth=UMAP_TICK_LINEWIDTH / scale)
    for value in ax.get_yticks():
        if ax.get_ylim()[0] <= value <= ax.get_ylim()[1]:
            point = ax.transData.transform([ax.get_xlim()[0], value]) * unit
            line([point, point + [-UMAP_TICK_LENGTH / scale, 0]], linewidth=UMAP_TICK_LINEWIDTH / scale)
    for artist in ax.lines:
        points = artist.get_transform().transform(artist.get_xydata()) * unit
        line(points, artist.get_color(), artist.get_linewidth(), artist.get_alpha())
    pdf.restoreState()

    begin_group("Text")
    pdf.setFillColorRGB(0, 0, 0)
    pdf.setFillAlpha(1)
    specs = []
    for index, artist in enumerate(texts):
        origin = artist.get_transform().transform(artist.get_position())
        angle = np.deg2rad(artist.get_rotation())
        cosine, sine = float(np.cos(angle)), float(np.sin(angle))
        layout = artist._get_layout(renderer)[1]
        text = pdf.beginText()
        text.setFont("Arial", artist.get_fontsize())
        for value, _, x, y in layout:
            point = (origin + [x, y]) * unit
            text.setTextTransform(cosine, sine, -sine, cosine, *point)
            text.textOut(value)
        pdf.drawText(text)
        first_value, _, first_x, first_y = layout[0]
        first_point = (origin + [first_x, first_y]) * unit
        align = artist.get_ha()
        advance = pdfmetrics.stringWidth(first_value, "Arial", artist.get_fontsize())
        alignment_shift = advance * (0.5 if align == "center" else 1 if align == "right" else 0)
        native_anchor = first_point + alignment_shift * np.array([cosine, sine])
        leading = abs(layout[1][3] - layout[0][3]) * unit if len(layout) > 1 else artist.get_fontsize() * 1.2
        specs.append({"name": f"Text-{index:03d}", "text": artist.get_text(),
                      "fontsize": artist.get_fontsize(), "angle": artist.get_rotation(),
                      "leading": float(leading), "align": align,
                      "anchor": native_anchor.tolist()})
    pdf.restoreState()
    pdf.showPage()
    pdf.save()
    native_text_pdf(base_path, destination, specs)

In [ ]:
#9. =========== Draw six final integration PDFs
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
generated_paths = []
for dataset in UMAP_INPUTS:
    xy = umap_arrays[dataset+"_xy"]
    for kind,suffix,palette in (("Cell","cell",UMAP_CELL_ANNOTATION_COLORS),("Stage","stage",UMAP_STAGE_COLORS)):
        keys = umap_arrays[dataset+"_"+suffix]
        category_order = [key for key in palette if key in set(keys)]
        assert sum(np.count_nonzero(keys == key) for key in category_order) == len(xy)
        fig,ax = umap_make_figure(dataset,kind,xy,keys,palette,category_order)
        if UMAP_USE_SAVED_LAYOUT:
            positions = umap_place_saved_labels(fig,ax,dataset,kind,xy,keys,category_order)
        else:
            positions = umap_place_labels(fig,ax,dataset,kind,xy,keys,category_order)
        # Set the output file path.
        destination = ROOT
        umap_export_pdf(fig,ax,destination,dataset)
        plt.close(fig)
        generated_paths.append(destination)
        print(f"{destination.name}: {len(xy):,} cells, {len(category_order)} categories; s={UMAP_POINT_SIZE[dataset]:g}, alpha={UMAP_POINT_ALPHA:g}")

## 10. DotPlot panel and appearance
The 54 unique genes form ten modules. Each Type I/II subtype contains three developmental markers, three mature markers and three exploratory DEGs. Developmental markers may be shared across precursor programs; assignment to a single column does not imply exclusivity. The same cell-label display rules and colors are used in the UMAPs and DotPlot.

In [ ]:
#10. =========== DotPlot parameters and selected marker panel
DOT_DOT_MAX_AREA = 36.0             # Original Matplotlib marker scale, pt^2; area is proportional to % Exp.
DOT_COLUMN_GAP_RATIO = 0.5          # Horizontal edge clearance divided by the largest data-dot diameter.
DOT_ROW_PITCH_PT = 10.5             # Row-center distance, pt.
DOT_ROW_FONT = 6.5
DOT_GENE_FONT = 6.3
DOT_GROUP_FONT = 6.5
DOT_TITLE_FONT = 10.0
DOT_LEGEND_FONT = 7.0
DOT_NOTE_FONT = 5.8
DOT_ROW_CIRCLE_DIAMETER_PT = 4.5
DOT_ROW_CIRCLE_ALPHA = 0.82         # Match the cell-UMAP legend markers.
DOT_ROW_CIRCLE_TEXT_GAP_PT = 4.0
DOT_ROW_MATRIX_GAP_PT = 9.0
DOT_GENE_ROTATION = 55.0
DOT_GROUP_ROTATION = 55.0
DOT_NARROW_GROUP_ROTATION = 90.0
DOT_GROUP_WRAP = 22
DOT_GROUP_LINEWIDTH_PT = 1.5
DOT_GROUP_TEXT_GAP_PT = 5.0
DOT_ROLE_FONT = 6.0
DOT_ROLE_BAND_HEIGHT_PT = 13.0
DOT_TEXT_LEADING_RATIO = 1.1
DOT_MARGIN_PT = 13.0
DOT_COLOR_MIN, DOT_COLOR_MAX = -1.0, 1.0
DOT_COLORBAR_WIDTH_PT = 151.2
DOT_COLORBAR_HEIGHT_PT = 5.4
DOT_COLORBAR_STEPS = 256
DOT_PURPLE_ANCHORS = ("#F1F1F1", "#E8C6DE", "#C67ABA", "#952E7D", "#69005F")
DOT_DISPLAY_REPLACEMENTS = UMAP_DISPLAY_REPLACEMENTS

# Selected 54-gene panel: two added programs and three exploratory DEGs per subtype.
# Original genes retain the saved 168-gene statistics and global calibration.
# Selection references: Shrestha et al., Cell 2018, doi:10.1016/j.cell.2018.07.007;
# Petitpre et al., Nature Communications 2022, doi:10.1038/s41467-022-31580-1;
# Shrestha et al., Developmental Cell 2023, doi:10.1016/j.devcel.2023.01.008.
# Immature includes validated shared precursor programs, not age-exclusive markers.
# Further validation: Sun et al., Cell 2018, doi:10.1016/j.cell.2018.07.008;
# Sanders and Kelley, PNAS 2022, doi:10.1073/pnas.2203935119.
# Otic/neurogenic and cycling markers: Matern et al. 2023 (PMC10592509),
# Sun et al., Cell Reports 2022, doi:10.1016/j.celrep.2022.110542.
# Eya1/Six1 functional validation: Ahmed et al. 2012 (PMC3347689).
# Myt1/Epha5/Shh/Prox1: Matern 2023, Sun 2022, Lu 2011 (PMC3167573).
# Ia: Tle4/Islr2 are shared precursors; Calb2 is validated before and after birth.
# Ib: Runx1/Igfbpl1 are shared developmental markers; Calb1 also persists in adults.
# Ic: Ntng1/Meis2 include the shared Ib/Ic path; Mgat4c has embryonic RNAscope validation.
# II: Gfra1 includes the shared Ia/II path; Efna5/Tac1 have perinatal projection validation.
# Ncald is Ib/Ic-shared (2023, Fig. S1H); it appears once in Type Ib to avoid duplicate columns.
DOT_MARKER_GROUPS = {'Otic origin': ['Six1', 'Eya1', 'Foxg1'],
 'Early neurogenesis': ['Neurog1', 'Neurod1', 'Insm1'],
 'Cycling': ['Mki67', 'Top2a'],
 'Neuronal differentiation': ['Myt1', 'Epha5', 'Shh', 'Prox1'],
 'Neuronal identity': ['Elavl3', 'Uchl1', 'Map2'],
 'Type Ia': ['Tle4', 'Islr2', 'Calb2', 'Rxrg', 'Pcdh20', 'B3gat1', 'Cpne4', 'Dok5', 'Tmem108'],
 'Type Ib': ['Runx1', 'Igfbpl1', 'Calb1', 'Ttn', 'Kcnd2', 'Ncald', 'Sema3e', 'Nrp2', 'Rgma'],
 'Type Ic': ['Ntng1', 'Mgat4c', 'Meis2', 'Lypd1', 'Pou4f1', 'Grm8', 'Bhlhe22', 'Tenm2', 'Cdk14'],
 'Type II': ['Gfra1', 'Efna5', 'Tac1', 'Ngfr', 'Prph', 'Th', 'St8sia4', 'Fnbp1l', 'Nog'],
 'Neuronal maturation': ['Nefh', 'Snap25', 'Slc17a7']}
DOT_MODULE_COLORS = {
    "Otic origin": "#5B7F9D", "Early neurogenesis": "#8B759C", "Cycling": "#B29A59",
    "Neuronal differentiation": "#5C9084", "Type Ia": "#B7864A", "Type Ib": "#B65F64",
    "Type Ic": "#557B9F", "Type II": "#519A94",
    "Neuronal identity": "#6F748C", "Neuronal maturation": "#377D8B",
}
DOT_TYPE_ROLE_COUNTS = {
    group: (("Immature", 3), ("Mature", 3), ("DEGs", len(DOT_MARKER_GROUPS[group]) - 6))
    for group in ("Type Ia", "Type Ib", "Type Ic", "Type II")
}

#3. =========== Exploratory DEG screening parameters
# Changing these settings rebuilds the reusable statistics, not the integration.
DOT_DE_STAGES = ("P1", "P7", "P28")
DOT_DE_TYPES = ("Type Ia", "Type Ib", "Type Ic", "Type II")
DOT_DE_TARGET_SUM = 10000.0
DOT_DE_RATIO_PSEUDOCOUNT = 0.1       # CP10k pseudocount in descriptive log2 ratios.
DOT_DE_FDR_THRESHOLD = 0.05
DOT_DE_MIN_TARGET_PCT = 25.0       # Equal-stage mean detection, percent.
DOT_DE_MIN_POSITIVE_STAGES = 3
DOT_DE_MIN_POSITIVE_LIBRARIES = 5
DOT_DE_SELECTION = {group: DOT_MARKER_GROUPS[group][6:] for group in DOT_DE_TYPES}
DOT_BASE_PANEL_GENES = {gene for group,members in DOT_MARKER_GROUPS.items()
                    for gene in (members[:6] if group in DOT_DE_TYPES else members)}
DOT_DE_SETTINGS = dict(stages=DOT_DE_STAGES,types=DOT_DE_TYPES,target_sum=DOT_DE_TARGET_SUM,
                   ratio_pseudocount=DOT_DE_RATIO_PSEUDOCOUNT,fdr=DOT_DE_FDR_THRESHOLD,
                   min_target_pct=DOT_DE_MIN_TARGET_PCT,min_positive_stages=DOT_DE_MIN_POSITIVE_STAGES,
                   min_positive_libraries=DOT_DE_MIN_POSITIVE_LIBRARIES,selection=DOT_DE_SELECTION,
                   positive_effects="Mean log2fc, meanlog difference and detection difference > 0")

# Functional evidence supports hypotheses; it does not establish subtype-specific cochlear mechanisms.
# Record the source model and the limits of the cochlear interpretation.

## 11. Functional evidence for exploratory genes
These entries preserve the source organisms, experiments and limitations used during selection. They support potential cochlear-development hypotheses; they do not establish subtype-specific causal functions in SGNs. No network connection is used when running this notebook.

In [ ]:
#11. =========== Recorded functional evidence
DOT_DE_FUNCTIONAL_EVIDENCE = {
    "Cpne4": dict(uniprot="https://www.uniprot.org/uniprotkb/Q8BLR2/entry",
        papers=["https://pmc.ncbi.nlm.nih.gov/articles/PMC8631636/"],
        evidence="Calcium-dependent membrane adaptor; overexpression induces dendritic varicosities in retinal ganglion cells and identifies cytoskeletal/vesicular partners.",
        hypothesis="Potential calcium-dependent membrane/cytoskeleton remodeling in Ia neurites.",
        boundary="Retinal overexpression evidence; normal dendrite or axon formation was not increased. SGN causal function is unvalidated."),
    "Dok5": dict(uniprot="https://www.uniprot.org/uniprotkb/Q91ZM9/entry",
        papers=["https://pubmed.ncbi.nlm.nih.gov/16647839/","https://pmc.ncbi.nlm.nih.gov/articles/PMC2150770/"],
        evidence="Adaptor for RET and TrkB/TrkC; experimentally participates in neurotrophin-induced MAPK signaling and neuronal differentiation in cell models.",
        hypothesis="Potential subtype-dependent neurotrophin response and neurite differentiation in Ia.",
        boundary="Mechanism established outside SGN; Ia-specific cochlear effects are a hypothesis."),
    "Tmem108": dict(uniprot="https://www.uniprot.org/uniprotkb/Q8BHE4/entry",
        papers=["https://pubmed.ncbi.nlm.nih.gov/21849472/","https://pubmed.ncbi.nlm.nih.gov/27605705/"],
        evidence="Retrolinkin regulates neuronal microtubule/retrograde transport and BDNF-TrkB endocytosis linked to dendritic growth.",
        hypothesis="Potential control of neurotrophin trafficking and Ia neurite maturation.",
        boundary="General neuronal mechanisms; SGN subtype-specific causal function is unvalidated."),
    "Sema3e": dict(uniprot="https://www.uniprot.org/uniprotkb/P70275/entry",
        papers=["https://www.nature.com/articles/nature08000"],
        evidence="Sema3E-PlexinD1 signaling controls sensory-motor synaptic specificity in genetic experiments.",
        hypothesis="Potential control of Ib target recognition and connectivity choice.",
        boundary="Evidence from a non-cochlear circuit. Do not represent SEMA3E as the NRP2 ligand."),
    "Nrp2": dict(uniprot="https://www.uniprot.org/uniprotkb/O35375/entry",
        papers=["https://pmc.ncbi.nlm.nih.gov/articles/PMC4566076/"],
        evidence="NRP2-SEMA3F repulsion restricts SGN branching and promotes inner hair cell innervation in the developing cochlea.",
        hypothesis="Potential subtype-dependent guidance sensitivity associated with Ib enrichment.",
        boundary="Cochlear guidance is established; a specific Ib developmental mechanism is unvalidated."),
    "Rgma": dict(uniprot="https://www.uniprot.org/uniprotkb/Q6PCX7/entry",
        papers=["https://pmc.ncbi.nlm.nih.gov/articles/PMC5540376/","https://pmc.ncbi.nlm.nih.gov/articles/PMC7859405/"],
        evidence="RGMA-NEO1/RhoA signaling regulates neurites; RGMA inhibition increases afferent synapse formation and repair in cochlear experiments.",
        hypothesis="Potential Ib-specific limits on branching, synaptogenesis or plasticity.",
        boundary="General cochlear evidence exists; the proposed Ib-specific role remains unvalidated."),
    "Bhlhe22": dict(uniprot="https://www.uniprot.org/uniprotkb/Q8C6A8/entry",
        papers=["https://pmc.ncbi.nlm.nih.gov/articles/PMC3269007/"],
        evidence="BHLHE22/BHLHB5 cooperates with PRDM8 in transcriptional repression and CDH11-associated neural circuit assembly; knockout causes axon mistargeting.",
        hypothesis="Potential transcriptional control of Ic differentiation and connectivity.",
        boundary="Mechanistic evidence from other neuronal populations; an SGN causal role is unvalidated."),
    "Tenm2": dict(uniprot="https://www.uniprot.org/uniprotkb/Q9WTS5/entry",
        papers=["https://elifesciences.org/articles/83276"],
        evidence="Transmembrane adhesion/synaptic organizer; hippocampal-neuron experiments link TEN2-microtubule interactions to synaptic receptor delivery.",
        hypothesis="Potential Ic connectivity recognition, synapse organization or receptor trafficking.",
        boundary="The cited functional model is hippocampal inhibitory synapses; the SGN application is a hypothesis."),
    "Cdk14": dict(uniprot="https://www.uniprot.org/uniprotkb/O35495/entry",
        papers=["https://doi.org/10.1186/s12915-023-01732-w"],
        evidence="PFTK1/CDK14 regulates developing cortical axon growth through phosphorylation and activation of RhoA.",
        hypothesis="Potential RhoA-dependent control of Ic axon growth and branching.",
        boundary="Postmitotic-neuron functional evidence supports selection; an SGN causal role is unvalidated."),
    "St8sia4": dict(uniprot="https://www.uniprot.org/uniprotkb/Q64692/entry",
        papers=["https://pmc.ncbi.nlm.nih.gov/articles/PMC6772332/"],
        evidence="Polysialyltransferase modifying NCAM-associated glycans; genetic evidence supports modulation of neural adhesion and synaptic plasticity.",
        hypothesis="Potential regulation of II neurite adhesion, remodeling and plasticity by surface polysialylation.",
        boundary="St8sia4 single knockout has grossly normal early brain structure with St8sia2 compensation; an SGN-specific function is unvalidated."),
    "Fnbp1l": dict(uniprot="https://www.uniprot.org/uniprotkb/Q8K012/entry",
        papers=["https://pubmed.ncbi.nlm.nih.gov/16885158/"],
        evidence="TOCA-1 links CDC42/N-WASP-associated actin remodeling and membrane tubulation; knockdown alters neurite extension and neuronal axon branching.",
        hypothesis="Potential control of the elongated, branching peripheral arbor of II neurons.",
        boundary="Functional data are from cultured cell and non-cochlear neuronal models; the II-specific mechanism is a hypothesis."),
    "Nog": dict(uniprot="https://www.uniprot.org/uniprotkb/P97466/entry",
        papers=["https://pmc.ncbi.nlm.nih.gov/articles/PMC5843465/"],
        evidence="Secreted BMP antagonist; Nog-null cochleae have additional inner and outer hair-cell rows.",
        hypothesis="Potential local BMP modulation associated with II development and neuron-epithelium interactions.",
        boundary="Direct cochlear evidence concerns sensory-organ patterning, not a proven II-neuron source or function."),
}

## 12. Read saved marker statistics and common counts
For the 41 genes present in the original 168-gene panel, reuse the integration model’s exact group means, detection rates and calibration. Aggregate the 13 added genes from the counts layer in `Whole_CSS.h5ad` using `log1p(count / sum of 9174 common genes × 10000)`. The inherited `marker_common_total_counts` observation field is not the whole-period denominator. Unmeasured gene/group entries remain missing and are shown as crosses.

In [ ]:
#12. =========== DotPlot input paths
# Set the result file path; uses Whole_CSS_model.joblib generated by whole-period CSS integration.
DOT_MODEL_PATH = ROOT
# Set the result file path; uses Whole_CSS.h5ad generated by whole-period CSS integration.
DOT_DATA_PATH = ROOT
DOT_PALETTE_PATH = UMAP_STATE_PATH
# Set the data file path; uses Fig1-dotplot.npz from final Fig1 plot preparation.
DOT_STATE_PATH = ROOT
# Set the output file path.
DOT_OUTPUT_PATH = ROOT
def dot_input_program():
    """Describe the reusable plotting state without reading expression matrices."""
    with np.load(DOT_PALETTE_PATH, allow_pickle=False) as bundle:
        palette = dict(UMAP_CELL_ANNOTATION_COLORS)
        source_labels = bundle["Whole_Period_cell"].copy()
        source_ids = bundle["Whole_Period_cell_ids"].copy()
    selected = [(group, gene) for group,members in DOT_MARKER_GROUPS.items() for gene in members]
    selected_genes = [gene for _,gene in selected]
    assert len(selected_genes) == len(set(selected_genes)) == 54, "Wrong or duplicate gene columns."
    assert len(DOT_MARKER_GROUPS) == 10 and set(DOT_MODULE_COLORS) == set(DOT_MARKER_GROUPS)
    assert set(DOT_DE_FUNCTIONAL_EVIDENCE) == {gene for members in DOT_DE_SELECTION.values() for gene in members}
    for group,roles in DOT_TYPE_ROLE_COUNTS.items():
        assert sum(count for _,count in roles) == len(DOT_MARKER_GROUPS[group])
        assert all(count >= 0 for _,count in roles)
    sources = {}
    for name,path in (("saved_contract",DOT_MODEL_PATH),("common_counts",DOT_DATA_PATH)):
        info = path.stat()
        sources[name] = dict(path=str(path.relative_to(ROOT)),size=info.st_size,mtime_ns=info.st_mtime_ns)
    program = dict(format="Fig1-dotplot-v3",gene_program=DOT_MARKER_GROUPS,
                   genes=selected_genes,modules=[group for group,_ in selected],
                   roles=DOT_TYPE_ROLE_COUNTS,palette=palette,module_colors=DOT_MODULE_COLORS,
                   display_replacements=DOT_DISPLAY_REPLACEMENTS,sources=sources,
                   de_settings=DOT_DE_SETTINGS,de_functional_evidence=DOT_DE_FUNCTIONAL_EVIDENCE,
                   source_ids_sha256=hashlib.sha256(source_ids.tobytes()).hexdigest(),
                   source_labels_sha256=hashlib.sha256(source_labels.tobytes()).hexdigest())
    signature = hashlib.sha256(json.dumps(program,sort_keys=True).encode("utf-8")).hexdigest()
    return program,signature,palette,source_labels,source_ids


def dot_build_inputs(program=None, signature=None, palette=None, source_labels=None, source_ids=None):
    """Reuse all existing statistics; aggregate only genes absent from the saved 168."""
    if program is None:
        program,signature,palette,source_labels,source_ids = dot_input_program()
    contract = joblib.load(DOT_MODEL_PATH)["figure_contract"]
    stats = contract["dot_stats"]
    levels = list(contract["dot_row_order"])
    with h5py.File(DOT_DATA_PATH, "r") as handle:
        genes = np.asarray(read_elem(handle["uns/css_marker_gene_symbols"]), dtype=str)
        modules = np.asarray(read_elem(handle["uns/css_marker_groups"]), dtype=str)
    counts = np.asarray(stats["n_cells"], dtype=int)
    mean = np.asarray(stats["mean"], dtype=float)
    pct = np.asarray(stats["pct"], dtype=float)
    coverage = np.asarray(stats["coverage"], dtype=float)
    center = np.asarray(contract["dot_center"], dtype=float)
    scale = np.asarray(contract["dot_scale"], dtype=float)
    assert levels == stats["levels"] and len(levels) == len(set(levels)) == 36
    assert set(levels) == set(palette) == set(source_labels)
    assert np.array_equal(counts, [np.count_nonzero(source_labels == level) for level in levels])
    assert len(genes) == len(set(genes)) == 168 and len(set(modules)) == 27
    assert pct.shape == mean.shape == coverage.shape == (36, 168)
    assert np.isfinite(pct[coverage > 0]).all() and np.isfinite(mean[coverage > 0]).all()
    assert ((pct[coverage > 0] >= 0) & (pct[coverage > 0] <= 100)).all()
    assert ((coverage >= 0) & (coverage <= 1)).all()
    assert np.allclose(center, np.nanmean(mean, axis=0))
    assert np.allclose(scale, np.nanstd(mean, axis=0, ddof=0))
    scaled = np.divide(mean - center, scale, out=np.zeros_like(mean), where=scale > 0)
    scaled[coverage == 0] = np.nan
    assert np.isfinite(scaled[coverage > 0]).all()
    selected_genes = program["genes"]
    original_index = {gene:i for i,gene in enumerate(genes)}
    selected_shape = (len(levels),len(selected_genes))
    result = dict(levels=levels,counts=counts,genes=np.asarray(selected_genes,dtype=str),
                  modules=np.asarray(program["modules"],dtype=str),palette=palette,module_colors=DOT_MODULE_COLORS)
    for key in ("mean","pct","coverage","scaled"):
        result[key] = np.full(selected_shape,np.nan,dtype=float)
    result["center"] = np.full(len(selected_genes),np.nan)
    result["scale"] = np.full(len(selected_genes),np.nan)
    old_arrays = dict(mean=mean,pct=pct,coverage=coverage,scaled=scaled,center=center,scale=scale)
    new_genes = [gene for gene in selected_genes if gene not in original_index]
    denominator_audit = None
    for j,gene in enumerate(selected_genes):
        if gene in original_index:
            old = original_index[gene]
            for key in ("mean","pct","coverage","scaled"):
                result[key][:,j] = old_arrays[key][:,old]
            result["center"][j],result["scale"][j] = center[old],scale[old]
    if new_genes:
        with h5py.File(DOT_DATA_PATH,"r") as handle:
            common_symbols = np.asarray(read_elem(handle["var/gene_symbol"]),dtype=str)
            obs_ids = np.asarray(read_elem(handle["obs/_index"]),dtype=str)
            labels = np.asarray(read_elem(handle["obs/source_stage_celltype"]),dtype=str)
            saved_totals = np.asarray(read_elem(handle["obs/marker_common_total_counts"]),dtype=float)
            raw_counts = read_elem(handle["layers/counts"])
            archived_raw = np.asarray(read_elem(handle["obsm/css_marker_raw_counts"]),dtype=float)
            archived_log = np.asarray(read_elem(handle["obsm/css_marker_log"]),dtype=float)
        assert raw_counts.shape == (8340,9174) and common_symbols.size == 9174
        assert len(set(common_symbols)) == len(common_symbols)
        assert np.array_equal(obs_ids,source_ids) and np.array_equal(labels,source_labels)
        assert np.isfinite(raw_counts.data).all() and (raw_counts.data >= 0).all()
        common_total = np.asarray(raw_counts.sum(axis=1,dtype=np.float64)).ravel()
        assert (common_total > 0).all()
        # The original Whole-period notebook uses this 9174-gene row sum.
        # marker_common_total_counts is inherited embryonic QC metadata and is missing postnatally.
        rebuilt_log = np.log1p(archived_raw * 10000 / common_total[:,None])
        assert np.array_equal(rebuilt_log,archived_log,equal_nan=True), "Common-count normalization does not reproduce the archived 168-gene log expression."
        finite_totals = np.isfinite(saved_totals)
        denominator_audit = dict(definition="Row sum of Whole_CSS layers/counts over all 9174 common genes.",
            archived_log_reproduction="Bit-exact for all 8340 cells and all 168 archived marker columns; NaN pattern preserved.",
            archived_finite_values=int(np.isfinite(archived_log).sum()),
            inherited_obs_field="marker_common_total_counts is embryonic source-QC metadata, not the Whole-period DotPlot denominator.",
            inherited_obs_missing=int((~finite_totals).sum()),
            inherited_obs_exact_matches=int(np.count_nonzero(common_total == saved_totals)),
            common_to_inherited_ratio_median=float(np.median(common_total[finite_totals]/saved_totals[finite_totals])),
            denominator_sha256=hashlib.sha256(common_total.tobytes()).hexdigest())
        lookup = {gene:i for i,gene in enumerate(common_symbols)}
        absent = set(new_genes) - set(lookup)
        if absent:
            raise ValueError("Genes absent from saved statistics and common counts: " + ", ".join(sorted(absent)))
        raw = raw_counts[:,[lookup[gene] for gene in new_genes]].toarray().astype(np.float64)
        normalized = np.log1p(raw * 10000 / common_total[:,None])
        new_mean = np.stack([normalized[labels == level].mean(axis=0) for level in levels])
        new_pct = np.stack([100 * np.count_nonzero(raw[labels == level] > 0,axis=0) / count
                            for level,count in zip(levels,counts)])
        new_center,new_scale = new_mean.mean(axis=0),new_mean.std(axis=0,ddof=0)
        new_scaled = np.divide(new_mean-new_center,new_scale,out=np.zeros_like(new_mean),where=new_scale > 0)
        for k,gene in enumerate(new_genes):
            j = selected_genes.index(gene)
            result["mean"][:,j],result["pct"][:,j] = new_mean[:,k],new_pct[:,k]
            result["coverage"][:,j] = 1.0
            result["scaled"][:,j] = new_scaled[:,k]
            result["center"][j],result["scale"][j] = new_center[k],new_scale[k]
    for j,gene in enumerate(selected_genes):
        if gene in original_index:
            old = original_index[gene]
            for key in ("mean","pct","coverage","scaled"):
                assert np.array_equal(result[key][:,j],old_arrays[key][:,old],equal_nan=True), "Saved statistics changed."
            assert result["center"][j] == center[old] and result["scale"][j] == scale[old]
    measured = result["coverage"] > 0
    assert np.isfinite(result["mean"][measured]).all() and np.isfinite(result["scaled"][measured]).all()
    assert ((result["pct"][measured] >= 0) & (result["pct"][measured] <= 100)).all()
    result["metadata"] = dict(program,input_signature=signature,
        method=dict(original="Exact saved figure_contract dot_stats, dot_center and dot_scale; no recomputation.",
                    supplemental="Common 9174-gene layers/counts; log1p(raw/common_total*10000); mean and positive-cell percentage per original label.",
                    supplemental_denominator="Whole_CSS layers/counts 9174-gene row sum; verified by bit-exact reproduction of all archived 168-gene log values.",
                    scaling="Gene-wise mean/std across all 36 original groups; ddof=0. Clip only rendered colors to [-1,1]."),
        supplemental_genes=new_genes,denominator_audit=denominator_audit,n_cells=int(counts.sum()),row_order=levels,
        statistic_source={gene:("common_counts" if gene in new_genes else "saved_168") for gene in selected_genes})
    result["metadata"]["appearance"] = dict(dot_max_area_pt2=DOT_DOT_MAX_AREA,column_gap_ratio=DOT_COLUMN_GAP_RATIO,
        row_pitch_pt=DOT_ROW_PITCH_PT,row_font=DOT_ROW_FONT,gene_font=DOT_GENE_FONT,group_font=DOT_GROUP_FONT,
        title_font=DOT_TITLE_FONT,legend_font=DOT_LEGEND_FONT,note_font=DOT_NOTE_FONT,
        row_circle_diameter_pt=DOT_ROW_CIRCLE_DIAMETER_PT,row_circle_alpha=DOT_ROW_CIRCLE_ALPHA,
        gene_rotation=DOT_GENE_ROTATION,group_rotation=DOT_GROUP_ROTATION,
        color_limits=[DOT_COLOR_MIN,DOT_COLOR_MAX],purple_anchors=DOT_PURPLE_ANCHORS)
    de_arrays,de_methods = dot_compute_de_screen(raw_counts,contract)
    result.update({"de_"+key:value for key,value in de_arrays.items()})
    result["metadata"]["differential_expression"] = de_methods
    return result

## 13. Reproduce the exploratory DEG screen
This reproduces the saved library-stratified cell-level screen across P1/P7/P28. Cells are not biological replicates. The three selected genes per subtype are potential research candidates, not validated cochlear subtype mechanisms. The full screen is computed in memory and the source state file is not rewritten.

In [ ]:
#13. =========== Exploratory differential-expression calculations
def dot_benjamini_hochberg(values):
    order = np.argsort(values,kind="stable")
    corrected = values[order] * len(values) / np.arange(1,len(values)+1)
    corrected = np.minimum.accumulate(corrected[::-1])[::-1]
    result = np.empty_like(values)
    result[order] = np.minimum(corrected,1.0)
    return result


def dot_compute_de_screen(raw_counts,contract):
    """Rank potential developmental candidates; cells are not biological replicates."""
    with h5py.File(DOT_DATA_PATH,"r") as handle:
        obs,var = read_elem(handle["obs"]),read_elem(handle["var"])
    post = obs.input_stage.astype(str).isin(DOT_DE_STAGES).to_numpy()
    obs = obs.loc[post]
    counts = raw_counts[post].tocsr()
    assert counts.shape == (985,9174)
    subtype = obs.source_stage_celltype.astype(str).str.split(" | ",regex=False).str[-1].to_numpy()
    stage,library = obs.input_stage.astype(str).to_numpy(),obs.library_id.astype(str).to_numpy()
    libraries = tuple(pd.unique(library))
    assert len(libraries) == 5 and set(subtype) == set(DOT_DE_TYPES)
    assert all(set(subtype[library == value]) == set(DOT_DE_TYPES) for value in libraries)
    assert all(len(set(stage[library == value])) == 1 for value in libraries)
    denominator = np.asarray(counts.sum(axis=1,dtype=np.float64)).ravel()
    raw = counts.toarray().astype(np.float64)
    cp10k = raw * DOT_DE_TARGET_SUM / denominator[:,None]
    logged,detected = np.log1p(cp10k),raw > 0
    symbols = np.asarray(var.gene_symbol,dtype=str)
    technical = np.zeros(len(var),dtype=bool)
    for name in ("mt_feature","ERCC_feature","technical_feature","rRNA_feature"):
        technical |= var[name].fillna(False).to_numpy(dtype=bool)
    technical |= pd.Series(symbols).str.match(r"(?i)^mt-|^ERCC[-_]|^Rpl\d|^Rps\d").to_numpy()
    eligible = ~technical

    def profile(selected):
        identities = subtype[selected]
        return tuple(np.stack([array[selected][identities == identity].mean(axis=0)
                               for identity in DOT_DE_TYPES])
                     for array in (cp10k,logged,detected))

    def effects(cp,log,pct=None):
        rest_cp = (cp.sum(axis=0,keepdims=True)-cp)/3
        rest_log = (log.sum(axis=0,keepdims=True)-log)/3
        fold = np.log2((cp+DOT_DE_RATIO_PSEUDOCOUNT)/(rest_cp+DOT_DE_RATIO_PSEUDOCOUNT))
        difference = log-rest_log
        if pct is None:
            return fold,difference
        rest_pct = (pct.sum(axis=0,keepdims=True)-pct)/3
        return fold,difference,pct-rest_pct,rest_pct

    centered_u = np.zeros((4,len(var)))
    total_variance = np.zeros_like(centered_u)
    library_fc,library_difference = [],[]
    library_counts = []
    for value in libraries:
        selected = library == value
        local,identities = logged[selected],subtype[selected]
        ranks = rankdata(local,axis=0,method="average")
        size = len(local)
        rank_ss = ((ranks-(size+1)/2)**2).sum(axis=0)
        for ti,identity in enumerate(DOT_DE_TYPES):
            target = identities == identity
            n1,n2 = int(target.sum()),int((~target).sum())
            u = ranks[target].sum(axis=0)-n1*(n1+1)/2
            centered_u[ti] += u-n1*n2/2
            total_variance[ti] += n1*n2/(size*(size-1))*rank_ss
        cp,log,_ = profile(selected)
        fc,difference = effects(cp,log)
        library_fc.append(fc)
        library_difference.append(difference)
        library_counts.append([int(np.count_nonzero(identities == t)) for t in DOT_DE_TYPES])
    library_fc,library_difference = np.stack(library_fc),np.stack(library_difference)
    z = np.divide(centered_u,np.sqrt(total_variance),out=np.zeros_like(centered_u),where=total_variance > 0)
    p = ndtr(-z)
    p[total_variance == 0] = 1.0
    q = np.ones_like(p)
    q[:,eligible] = dot_benjamini_hochberg(p[:,eligible].ravel()).reshape(4,-1)
    stage_cp,stage_log,stage_pct,stage_fc,stage_diff,stage_pct_diff,rest_pct = [],[],[],[],[],[],[]
    for value in DOT_DE_STAGES:
        cp,log,pct = profile(stage == value)
        pct *= 100
        fc,difference,pct_difference,reference_pct = effects(cp,log,pct)
        for target,array in zip((stage_cp,stage_log,stage_pct,stage_fc,stage_diff,stage_pct_diff,rest_pct),
                                (cp,log,pct,fc,difference,pct_difference,reference_pct)):
            target.append(array)
    stage_cp,stage_log,stage_pct,stage_fc,stage_diff,stage_pct_diff,rest_pct = map(np.stack,
        (stage_cp,stage_log,stage_pct,stage_fc,stage_diff,stage_pct_diff,rest_pct))
    positive_stages = ((stage_fc > 0)&(stage_diff > 0)).sum(axis=0)
    positive_libraries = ((library_fc > 0)&(library_difference > 0)).sum(axis=0)
    avg_fc,avg_diff,avg_pct_diff = stage_fc.mean(axis=0),stage_diff.mean(axis=0),stage_pct_diff.mean(axis=0)
    target_pct = stage_pct.mean(axis=0)
    exclusion = {gene for group,members in contract["marker_groups"].items()
                 if group.startswith("Exclusion:") for gene in members}
    allowed = ~np.isin(symbols,list(exclusion|DOT_BASE_PANEL_GENES))
    allowed &= ~pd.Series(symbols).duplicated(keep=False).to_numpy()
    allowed &= var.target_biotype.astype(str).eq("protein_coding").to_numpy()
    passed = ((q < DOT_DE_FDR_THRESHOLD)&(target_pct >= DOT_DE_MIN_TARGET_PCT)
              &(positive_stages >= DOT_DE_MIN_POSITIVE_STAGES)&(positive_libraries >= DOT_DE_MIN_POSITIVE_LIBRARIES)
              &(avg_fc > 0)&(avg_diff > 0)&(avg_pct_diff > 0)&eligible[None]&allowed[None])
    selection = {}
    for ti,identity in enumerate(DOT_DE_TYPES):
        selection[identity] = []
        assert len(DOT_DE_SELECTION[identity]) == 3
        for gene in DOT_DE_SELECTION[identity]:
            j = int(np.flatnonzero(symbols == gene).item())
            assert passed[ti,j], f"Selected DEG fails the frozen criteria: {identity}, {gene}"
            selection[identity].append(dict(gene=gene,q=float(q[ti,j]),log2fc=float(avg_fc[ti,j]),
                meanlog_difference=float(avg_diff[ti,j]),pct_in=float(target_pct[ti,j]),
                pct_rest=float(rest_pct[:,ti,j].mean()),pct_difference=float(avg_pct_diff[ti,j]),
                stage_log2fc=stage_fc[:,ti,j].tolist(),library_log2fc=library_fc[:,ti,j].tolist()))
    arrays = dict(gene_id=np.asarray(var.index,dtype=str),gene=symbols,eligible=eligible,candidate_allowed=allowed,
                  p=p,q=q,z=z,stage_mean_CP10k=stage_cp,stage_mean_log=stage_log,stage_pct=stage_pct,
                  stage_log2fc=stage_fc,stage_meanlog_difference=stage_diff,stage_pct_difference=stage_pct_diff,
                  library_log2fc=library_fc,library_meanlog_difference=library_difference,
                  positive_stages=positive_stages,positive_libraries=positive_libraries,passed=passed)
    methods = dict(scope="985 final postnatal cells; unchanged common 9174-gene RNA axis",
        stages=DOT_DE_STAGES,libraries=libraries,subtypes=DOT_DE_TYPES,library_subtype_counts=library_counts,
        tested_genes=int(eligible.sum()),tests=int(eligible.sum()*4),
        test="One-sided positive, library-stratified Mann-Whitney/Wilcoxon; sum centered U / sqrt(sum tie-corrected variance); no continuity correction",
        correction="BH jointly across all eligible genes and all four subtype comparisons",
        inference="Exploratory cell-level subtype enrichment, not biological-replicate population DE",
        normalization="log1p(raw*10000/sum raw across unchanged 9174 common genes)",
        descriptive_effect="Equal mean of three stages; reference within each stage is equal mean of the other three subtypes. Rank test uses pooled rest cells within each library.",
        settings=DOT_DE_SETTINGS,selected=selection,
        selection_basis="Positive differential expression plus sourced developmental-function hypotheses; not solely statistical rank.",
        functional_evidence=DOT_DE_FUNCTIONAL_EVIDENCE,
        limitations="Annotations partly use these data; no independent validation. Library IDs do not establish independent biological replicates; P7 has one unverified library.")
    return arrays,methods

## 14. DotPlot layout and vector export
Dot area shows the percentage of expressing cells; the color scale uses each gene’s original 36-group mean/standard-deviation calibration, with clipping to [-1, 1] only during drawing. Column pitch is 1.5 times the maximum data-dot diameter.

In [ ]:
#14. =========== DotPlot layout
def dot_rgb(value):
    return tuple(int(value.lstrip("#")[i:i + 2], 16) / 255 for i in (0, 2, 4))


def dot_purple(value):
    # Match the original Matplotlib 256-entry LinearSegmentedColormap exactly.
    normalized = np.clip((float(value) - DOT_COLOR_MIN) / (DOT_COLOR_MAX - DOT_COLOR_MIN), 0, 1)
    t = min(int(normalized * 256), 255) / 255 * 4
    lo = min(int(t), 3)
    return np.asarray(dot_rgb(DOT_PURPLE_ANCHORS[lo])) * (1 - (t - lo)) + np.asarray(dot_rgb(DOT_PURPLE_ANCHORS[lo + 1])) * (t - lo)


def dot_display_label(key, count):
    for old, new in DOT_DISPLAY_REPLACEMENTS.items():
        key = key.replace(old, new)
    return f"{key} (n={count:,})"


def dot_text_spec(name, text, fontsize, x, y, align="left", angle=0.0):
    return dict(name=name, text=text, fontsize=float(fontsize), angle=float(angle),
                leading=float(fontsize * DOT_TEXT_LEADING_RATIO), align=align,
                anchor=[float(x), float(y)])


def dot_text_polygons(spec):
    widths = [pdfmetrics.stringWidth(line, "Arial", spec["fontsize"]) for line in spec["text"].split("\n")]
    shift = 0.5 if spec["align"] == "center" else 1.0 if spec["align"] == "right" else 0.0
    ascent, descent = pdfmetrics.getAscentDescent("Arial", spec["fontsize"])
    polygons = []
    angle = math.radians(spec["angle"])
    matrix = np.array([[math.cos(angle), -math.sin(angle)], [math.sin(angle), math.cos(angle)]])
    for i, width in enumerate(widths):
        x0, x1 = -shift * width, (1 - shift) * width
        y0, y1 = descent - i * spec["leading"], ascent - i * spec["leading"]
        polygons.append(np.array([[x0, y0], [x1, y0], [x1, y1], [x0, y1]]) @ matrix.T + spec["anchor"])
    return polygons


def dot_text_box(spec):
    points = np.concatenate(dot_text_polygons(spec))
    return np.r_[points.min(axis=0), points.max(axis=0)]


def dot_overlaps(a, b, pad=0.0):
    return a[0] < b[2] + pad and a[2] > b[0] - pad and a[1] < b[3] + pad and a[3] > b[1] - pad


def dot_text_overlaps(a, b):
    """Use rotated Arial rectangles; AABBs overstate collisions between angled genes."""
    if not dot_overlaps(dot_text_box(a),dot_text_box(b)):
        return False
    for pa in dot_text_polygons(a):
        for pb in dot_text_polygons(b):
            separated = False
            for polygon in (pa,pb):
                for edge in np.diff(polygon,axis=0)[:2]:
                    normal = np.array([-edge[1],edge[0]])
                    xa,xb = pa @ normal,pb @ normal
                    if xa.max() <= xb.min() or xb.max() <= xa.min():
                        separated = True
                        break
                if separated:
                    break
            if not separated:
                return True
    return False


def dot_make_layout(data):
    n_rows, n_genes = data["pct"].shape
    dmax = math.sqrt(DOT_DOT_MAX_AREA * np.nanmax(data["pct"]) / 100)
    pitch = (1 + DOT_COLUMN_GAP_RATIO) * dmax
    row_pitch = max(DOT_ROW_PITCH_PT, dmax + 1, DOT_ROW_FONT + 2)
    labels = [dot_display_label(key, count) for key, count in zip(data["levels"], data["counts"])]
    label_x = DOT_MARGIN_PT + DOT_ROW_CIRCLE_DIAMETER_PT + DOT_ROW_CIRCLE_TEXT_GAP_PT
    left = label_x + max(pdfmetrics.stringWidth(label, "Arial", DOT_ROW_FONT) for label in labels) + DOT_ROW_MATRIX_GAP_PT
    matrix_width, matrix_height = n_genes * pitch, n_rows * row_pitch
    # Measure rotated gene labels to set the bottom margin.
    partial = np.any(data["coverage"] < 1, axis=0)
    names = [gene + ("\u2020" if flag else "") for gene, flag in zip(data["genes"], partial)]
    gene_specs = [dot_text_spec(f"Gene-{i:03d}", name, DOT_GENE_FONT, left + (i + .5) * pitch, 0,
                            align="right", angle=DOT_GENE_ROTATION) for i, name in enumerate(names)]
    gene_depth = -min(dot_text_box(spec)[1] for spec in gene_specs)
    bottom = DOT_MARGIN_PT + 40 + gene_depth + 4
    top_matrix = bottom + matrix_height
    texts = []
    for i, label in enumerate(labels):
        y = top_matrix - (i + .5) * row_pitch
        texts.append(dot_text_spec(f"Row-{i:02d}", label, DOT_ROW_FONT, label_x, y - DOT_ROW_FONT * .27))
    for spec in gene_specs:
        spec["anchor"][1] = bottom - 4
        texts.append(spec)
    groups = data["modules"]
    starts = np.r_[0, np.flatnonzero(groups[1:] != groups[:-1]) + 1]
    ends = np.r_[starts[1:], n_genes]
    assert len(starts) == len(set(groups))
    headings, bars, role_specs = [], [], []
    for i, (start, end) in enumerate(zip(starts, ends)):
        group = groups[start]
        title = group if end - start < 4 else textwrap.fill(group, width=DOT_GROUP_WRAP,
                    break_long_words=False, break_on_hyphens=False)
        angle = DOT_NARROW_GROUP_ROTATION if end - start < 4 else DOT_GROUP_ROTATION
        spec = dot_text_spec(f"Module-{i:02d}", title, DOT_GROUP_FONT,
                         left + (start + end) * pitch / 2, top_matrix + 12,
                         align="center", angle=angle)
        spec["anchor"][1] += top_matrix + 9 + DOT_ROLE_BAND_HEIGHT_PT + DOT_GROUP_TEXT_GAP_PT - dot_text_box(spec)[1]
        # Stagger overlapping headings without changing bar or gene positions.
        for _ in range(200):
            box = dot_text_box(spec)
            if not any(dot_overlaps(box, dot_text_box(other), pad=1.5) for other in headings):
                break
            spec["anchor"][1] += 2
        else:
            raise ValueError("Module headings do not fit.")
        headings.append(spec)
        bars.append((left + start * pitch + .6, left + end * pitch - .6,
                     top_matrix + 8 + DOT_ROLE_BAND_HEIGHT_PT, data["module_colors"][group]))
        if group in DOT_TYPE_ROLE_COUNTS:
            cursor = start
            for role_index,(role,count) in enumerate(DOT_TYPE_ROLE_COUNTS[group]):
                if count == 0:
                    continue
                x = left + (cursor + count / 2) * pitch
                role_specs.append(dot_text_spec(f"Role-{i:02d}-{role_index}", role, DOT_ROLE_FONT,
                                           x, top_matrix + 7, align="center"))
                cursor += count
            assert cursor == end
    texts.extend(headings)
    texts.extend(role_specs)
    heading_top = max(dot_text_box(spec)[3] for spec in headings)
    legend_y = heading_top + 18
    color_x = left
    texts.append(dot_text_spec("Legend-AvgExp", "Avg. Exp.", DOT_LEGEND_FONT, color_x, legend_y + 13))
    for i, value in enumerate(np.linspace(DOT_COLOR_MIN, DOT_COLOR_MAX, 5)):
        texts.append(dot_text_spec(f"ColorTick-{i}", f"{value:.1f}", DOT_LEGEND_FONT,
                     color_x + DOT_COLORBAR_WIDTH_PT * i / 4, legend_y - 10, align="center"))
    size_x = color_x + DOT_COLORBAR_WIDTH_PT + 35
    texts.append(dot_text_spec("Legend-PctExp", "% Exp.", DOT_LEGEND_FONT, size_x, legend_y + 13))
    size_circles = []
    for i, value in enumerate((0, 25, 50, 75, 100)):
        x = size_x + i * 32
        radius = math.sqrt(DOT_DOT_MAX_AREA * value / 100) / 2
        size_circles.append((x, legend_y + DOT_COLORBAR_HEIGHT_PT / 2, radius))
        texts.append(dot_text_spec(f"SizeTick-{i}", str(value), DOT_LEGEND_FONT, x, legend_y - 10, align="center"))
    notes = ["Color: gene-wise z-score across all 36 annotated types (ddof=0); shared scaling.",
             "\u2020 Not measured in every source; % Exp. among measured cells. \u00d7 No measured cells.",
             "Immature includes shared precursor programs; markers may persist into maturity.",
             "DEGs: exploratory postnatal subtype enrichment; library-stratified Wilcoxon, joint BH q < 0.05."]
    for i, note in enumerate(notes):
        texts.append(dot_text_spec(f"Note-{i}", note, DOT_NOTE_FONT, left, DOT_MARGIN_PT + (len(notes)-1-i) * 9))
    right = max(left + matrix_width, max(dot_text_box(spec)[2] for spec in texts)) + DOT_MARGIN_PT
    height = legend_y + 50
    texts.append(dot_text_spec("Title", "Whole period: Marker expression", DOT_TITLE_FONT,
                           left + matrix_width / 2, height - DOT_MARGIN_PT, align="center"))
    # Use Arial font metrics to check text bounds and overlaps.
    for spec in texts:
        box = dot_text_box(spec)
        assert 0 <= box[0] < box[2] <= right and 0 <= box[1] < box[3] <= height, spec["name"]
    for members in (texts[:n_rows], headings, role_specs, gene_specs):
        assert not any(dot_text_overlaps(a,b) for i,a in enumerate(members) for b in members[i+1:]), "Text collision."
    assert row_pitch > dmax and np.isclose(pitch / dmax - 1, DOT_COLUMN_GAP_RATIO)
    return dict(width=right, height=height, left=left, bottom=bottom,
                matrix_width=matrix_width, matrix_height=matrix_height, top_matrix=top_matrix,
                pitch=pitch, row_pitch=row_pitch, dmax=dmax, texts=texts, bars=bars,
                colorbar=(color_x, legend_y), size_circles=size_circles,
                row_circle_x=DOT_MARGIN_PT + DOT_ROW_CIRCLE_DIAMETER_PT / 2)

In [ ]:
#15. =========== DotPlot vector PDF export
def dot_export_pdf(data, layout):
    TASK_TMP.mkdir(parents=True,exist_ok=True)
    os.environ["TMP"] = os.environ["TEMP"] = str(TASK_TMP)
    width, height = layout["width"], layout["height"]
    # Set the output file path.
    base_path = ROOT
    pdf = canvas.Canvas(str(base_path), pagesize=(width, height), pageCompression=1,
                        initialFontName="Arial", initialFontSize=7)
    pdf.setTitle("Whole period: Marker expression")

    def begin_group(name):
        pdf.saveState()
        pdf.addLiteral("% " + name)
        path = pdf.beginPath()
        path.rect(0, 0, width, height)
        pdf.clipPath(path, stroke=0, fill=0, fillMode=1)

    def circle(x, y, radius, face, opacity=1):
        pdf.setFillColorRGB(*face)
        pdf.setFillAlpha(opacity)
        pdf.circle(float(x), float(y), float(radius), stroke=0, fill=1)

    begin_group("Graphics")
    actual_diameters = []
    for i, key in enumerate(data["levels"]):
        y = layout["top_matrix"] - (i + .5) * layout["row_pitch"]
        circle(layout["row_circle_x"], y, DOT_ROW_CIRCLE_DIAMETER_PT / 2,
               dot_rgb(data["palette"][key]), DOT_ROW_CIRCLE_ALPHA)
        for j in range(len(data["genes"])):
            if data["coverage"][i,j] <= 0 or data["pct"][i,j] <= 0:
                continue
            diameter = math.sqrt(DOT_DOT_MAX_AREA * data["pct"][i,j] / 100)
            actual_diameters.append(diameter)
            circle(layout["left"] + (j + .5) * layout["pitch"], y,
                   diameter / 2, dot_purple(data["scaled"][i,j]))
    for x,y,radius in layout["size_circles"]:
        if radius > 0:
            circle(x,y,radius,(.5,.5,.5))
    pdf.restoreState()

    begin_group("Axes/Guides")
    pdf.setFillAlpha(1)
    pdf.setStrokeAlpha(1)
    pdf.setLineCap(0)
    for x0,x1,y,face in layout["bars"]:
        pdf.setStrokeColorRGB(*dot_rgb(face))
        pdf.setLineWidth(DOT_GROUP_LINEWIDTH_PT)
        pdf.line(float(x0),float(y),float(x1),float(y))
    pdf.setStrokeColorRGB(.67,.67,.67)
    pdf.setLineWidth(.4)
    arm = math.sqrt(DOT_DOT_MAX_AREA * .30) / 2
    for i,j in np.argwhere(data["coverage"] == 0):
        x = layout["left"] + (j + .5) * layout["pitch"]
        y = layout["top_matrix"] - (i + .5) * layout["row_pitch"]
        pdf.line(float(x-arm),float(y-arm),float(x+arm),float(y+arm))
        pdf.line(float(x-arm),float(y+arm),float(x+arm),float(y-arm))
    x,y = layout["colorbar"]
    for i in range(DOT_COLORBAR_STEPS):
        lo = x + i * DOT_COLORBAR_WIDTH_PT / DOT_COLORBAR_STEPS
        hi = x + (i+1) * DOT_COLORBAR_WIDTH_PT / DOT_COLORBAR_STEPS
        value = DOT_COLOR_MIN + (i+.5) / DOT_COLORBAR_STEPS * (DOT_COLOR_MAX-DOT_COLOR_MIN)
        pdf.setFillColorRGB(*dot_purple(value))
        pdf.rect(float(lo),float(y),float(hi-lo),DOT_COLORBAR_HEIGHT_PT,stroke=0,fill=1)
    pdf.setStrokeColorRGB(0,0,0)
    pdf.setLineWidth(.4)
    for i in range(5):
        px = x + DOT_COLORBAR_WIDTH_PT * i / 4
        pdf.line(float(px),float(y),float(px),float(y-2))
    pdf.restoreState()

    begin_group("Text")
    pdf.setFillColorRGB(0,0,0)
    pdf.setFillAlpha(1)
    for spec in layout["texts"]:
        angle = math.radians(spec["angle"])
        c,s = math.cos(angle),math.sin(angle)
        text = pdf.beginText()
        text.setFont("Arial",spec["fontsize"])
        for i,line in enumerate(spec["text"].split("\n")):
            advance = pdfmetrics.stringWidth(line,"Arial",spec["fontsize"])
            shift = advance * (.5 if spec["align"] == "center" else 1 if spec["align"] == "right" else 0)
            local_x,local_y = -shift,-i*spec["leading"]
            x = spec["anchor"][0] + c*local_x - s*local_y
            y = spec["anchor"][1] + s*local_x + c*local_y
            text.setTextTransform(c,s,-s,c,x,y)
            text.textOut(line)
        pdf.drawText(text)
    pdf.restoreState()
    pdf.showPage()
    pdf.save()
    # Verify the sizes of data circles, excluding legend symbols.
    actual_dmax = max(actual_diameters)
    assert np.isclose(actual_dmax,layout["dmax"])
    assert np.isclose((layout["pitch"]-actual_dmax)/actual_dmax,DOT_COLUMN_GAP_RATIO)
    assert layout["row_pitch"] > actual_dmax
    native_text_pdf(base_path,DOT_OUTPUT_PATH,layout["texts"])
    print(f"Full DotPlot: {len(data['levels'])} rows, {len(data['genes'])} genes, {len(set(data['modules']))} modules.")
    print(f"Physical data-dot diameter={actual_dmax:.3f} pt; column pitch={layout['pitch']:.3f} pt; edge-gap ratio={DOT_COLUMN_GAP_RATIO:.3f}.")
    print(f"Measured-zero points omitted; unmeasured cells shown as crosses: {np.count_nonzero(data['coverage']==0)}.")

In [ ]:
#16. =========== Rebuild and verify the final 54-gene statistics
dot_data = dot_build_inputs()
with np.load(DOT_STATE_PATH,allow_pickle=False) as reference:
    for key in reference.files:
        if key == "metadata_json":
            continue
        value = np.asarray(dot_data[key])
        same = (np.array_equal(value,reference[key],equal_nan=True)
                if value.dtype.kind in "fc" else np.array_equal(value,reference[key]))
        assert same, f"Saved DotPlot statistics differ: {key}"
print(f"Verified {len(dot_data['levels'])} original groups, {len(dot_data['genes'])} unique genes and the full exploratory DEG arrays.")

In [ ]:
#17. =========== Draw the complete final DotPlot
dot_layout = dot_make_layout(dot_data)
dot_export_pdf(dot_data,dot_layout)
generated_paths.append(DOT_OUTPUT_PATH)

## 18. Output checks and cleanup
Verify embedded Arial, Unicode mapping, semantic editable text objects, vector paths and the expected flat clipping structure, with no image or Form XObjects. The two Illustrator-resaved embryonic presentations have zero clips; the other five have three sibling clips. The source integration files and saved plot states are read-only.

During preparation, this notebook was executed from top to bottom to a temporary output directory. All seven regenerated pages were compared against the current final folder at 144 and 300 DPI. PDF metadata and internal object numbering can differ; visible pixels, cell data, labels and plotting parameters are the reproduction criteria.

### Recorded reproduction check (2026-10-21)

- Clean `scvi_env` execution: all 18 code cells passed. All 54-gene statistics and the full exploratory DEG arrays exactly matched the saved state.
- All seven vector PDFs passed the figure validator: embedded Arial, explicit Unicode mapping, no raster images or Form XObjects; two embryonic pages have zero clips, the other five have three sibling clips of depth one.
- At both 144 and 300 DPI, the five postnatal/whole-period pages matched the current folder pixel for pixel. The two unnormalized Illustrator-native embryonic exports also matched their current pages pixel for pixel.
- Normalizing the two embryonic exports preserves their geometry, color, opacity, Adobe kerning and editable Illustrator artwork. Poppler renders a small number of transparent dot overlaps and thin leader edges differently after the transparency Forms are expanded. Pixels with a channel difference greater than 8/255 were: Cell 0.07822% at 144 DPI and 0.04407% at 300 DPI; Stage 0.00150% and 0.000108%. Isolated text renders were pixel-identical at both resolutions. This is the remaining standard-PDF preview limitation, not a change to labels or plotting parameters.
- The final normalized PDFs were reopened in Illustrator: all 90 embryonic semantic text frames, Arial fonts and text anchors matched the current final artwork. The current seven reference PDFs were not overwritten during verification.

For future tuning, edit the UMAP palette once; the DotPlot uses that same palette and display-name mapping. Label-offset search settings apply when `UMAP_USE_SAVED_LAYOUT=False`. After rerunning plotting cells, rerun the final check/cleanup cell.


In [ ]:
#18. =========== Check generated PDF structure and remove temporary exports
# Set the output file path.
generated_paths = [ROOT
                   for dataset in UMAP_INPUTS for kind in ("Cell","Stage")]
generated_paths.append(DOT_OUTPUT_PATH)
for path in generated_paths:
    reader = PdfReader(path)
    assert len(reader.pages) == 1
    page = reader.pages[0]
    operations = ContentStream(page.get_contents(),reader).operations
    operators = [op for _,op in operations]
    expected_clips = 0 if path.stem in {"Embryonic_Cell","Embryonic_Stage"} else 3
    assert operators.count(b"W") == expected_clips and b"W*" not in operators
    assert operators.count(b"BT") > 1 and operators.count(b"c") > 0
    resources = page["/Resources"]
    assert not resources.get("/XObject")
    for font_ref in resources["/Font"].values():
        font = font_ref.get_object()
        assert "Arial" in str(font["/BaseFont"]) and "/ToUnicode" in font
        assert "/FontFile2" in font["/FontDescriptor"].get_object()
    assert "/Illustrator" in page["/PieceInfo"]
    print(f"PASS {path.name}: vector artwork, embedded Arial, editable native text; {expected_clips} clips.")
assert len(generated_paths) == 7 and len(set(generated_paths)) == 7
task_path = TASK_TMP.resolve()
assert task_path.parent == WORK_ROOT.resolve() and task_path.name.startswith("run-")
shutil.rmtree(task_path)
for key,value in ORIGINAL_TEMP_ENV.items():
    if value is None:
        os.environ.pop(key,None)
    else:
        os.environ[key] = value
tempfile.tempdir = None
print("Completed seven PDFs; temporary export files removed.")